# KV-cache compression with ILF-diversified selection and mixed precision

This notebook implements and evaluates the project proposal end to end:

| Proposal item | Where |
|---|---|
| Attention scores $a_{i,l}$, layer frequency $d_i$, $\mathrm{ILF}_i$, diversified score $S_{i,l}=a_{i,l}(1+\lambda\,\mathrm{ILF}_i)$ | §3 and §4 |
| Dynamic selection (evict tokens) with plain attention ($\lambda=0$) vs ILF ($\lambda>0$) | §4 |
| ILF-guided mixed precision, {16, 8, 4, 0} bits per token, with **real packed storage** | §4 and §5 |
| Matched memory budgets plus uniform-quantization baselines and ablations | §8 |
| MMLU (general reasoning) | §10 |
| LongBench (long context) | §11 |
| Long-range retrieval with a **reused** prefix cache (many queries per compressed prefix) | §12 |
| Bytes-retained %, measured GPU memory, decode latency at batch 4 × 4096 tokens | §13 |
| Unique retained positions across layers (diversity), λ sweep | §14 |
| Follow-up experiments E0–E6: why ILF does not help, and what does (layer coverage, scoring signal, haystack, bit allocation) | §14b |
| Several models (Qwen 2.5, Llama 2, Llama 3) | §16 |

**How to run.** Use a Colab GPU runtime and run top to bottom. `SMOKE_TEST = True` (§2) runs
every experiment on a handful of examples in a few minutes to check the pipeline. Set it to
`False` for real numbers. Results are saved as CSV/PNG under `results/<model>/`.

**Implementation notes.**
* A custom attention backend (`kv_compress`, SDPA-based) records $a_{i,l}$ during prefill:
  the attention mass that the last `OBS_WINDOW` prefix queries put on each token, averaged
  over heads (SnapKV-style). That avoids `output_attentions`, whose quadratic memory does not
  fit at 4k tokens. The backend also builds its own causal mask, so it stays correct after
  eviction, when cache length ≠ position index.
* Keys are cached after RoPE, and new tokens are fed with their *original* position ids, so
  eviction never shifts positions.
* The first `SINK_TOKENS` and last `OBS_WINDOW` prefix tokens always stay at 16 bits; only
  the tokens between them are scored, evicted or quantized. Their bytes count against the budget.
* Quantization is asymmetric min-max with fp16 scale and zero: per channel over 32-token groups for keys
  (keys have outlier channels, as in KIVI), per token over 32-channel groups for values. 4-bit
  codes are packed two per byte. Quantized tokens are dequantized on the fly at every decode step.
* **Prefix-caching setup.** With `QUERY_AWARE = False`, LongBench prompts are split where the
  question starts. The document prefix is compressed without seeing the question, and the
  question is then run on top of the compressed cache. This matches the proposal's
  motivation (one cached prefix serving many future queries). Set `True` for the usual
  query-aware SnapKV-style protocol.


## 1. Dependencies
If installing changes a package already imported in your kernel, restart the kernel before continuing.


In [ ]:
%pip install -q "torch>=2.6" "transformers==4.57.1" "accelerate>=0.26.0" "lm_eval[hf,longbench]==0.4.11" sentencepiece pandas matplotlib

import json, math, time, random, re, gc
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import pandas as pd
from importlib.metadata import version

print({name: version(name) for name in ["torch", "transformers", "lm_eval"]})


## 2. Experiment configuration
`BUDGETS` are fractions of the fp16 KV-cache bytes. `"int4"` / `"int8"` resolve, for each input, to
exactly the bytes that uniform 4/8-bit quantization uses, which gives byte-matched comparisons.


In [ ]:
SMOKE_TEST = True   # tiny settings that run end to end in minutes; set False for real experiments

MODEL_ID = "Qwen/Qwen2.5-0.5B"
# Generalisation sweep (final section). Llama checkpoints are gated: request access on
# Hugging Face and run `from huggingface_hub import login; login()` first.
EXTRA_MODEL_IDS = ["Qwen/Qwen2.5-1.5B", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7b-hf"]

# Compression hyperparameters
SINK_TOKENS = 4
OBS_WINDOW = 32        # last W prefix tokens: always kept at 16 bits, and their queries give a_{i,l}
POOL_KERNEL = 7
LAMBDA = 1.0           # main ILF weight; LAMBDA_SWEEP explores it
LAMBDA_SWEEP = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
TIER_SPLIT = (1 / 3, 1 / 3, 1 / 3)  # budget share for the 16 / 8 / 4-bit tiers in mixed precision
# Memory budgets as a fraction of fp16 KV bytes. "int4"/"int8" match uniform quantization exactly.
BUDGETS = [0.1, 0.2, "int4", "int8"]
QUERY_AWARE = False    # False: compress the prefix before the question is seen (prefix caching)

MAX_INPUT_TOKENS = 4096
MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT = ["mmlu"], 10, 5   # limit applies per subject (57 subjects)
LONGBENCH_TASKS = ["longbench_hotpotqa", "longbench_2wikimqa", "longbench_multifieldqa_en",
                   "longbench_qasper", "longbench_passage_retrieval_en"]
LONGBENCH_LIMIT = 50
RETRIEVAL_DOCS, RETRIEVAL_FACTS, RETRIEVAL_TOKENS = 10, 20, 3000
BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS = 4, 4096, 64

# ---- Follow-up experiments (§14b). They need §1-§9 only; each RUN_E* flag runs one experiment. ----
RUN_E0 = RUN_E1 = RUN_E2 = RUN_E3 = RUN_E4 = RUN_E5 = RUN_E6 = True
EXP_DOCS = 30                  # retrieval documents per arm (seeds 0-9 are the ten used in §12)
EXP_GENERATE = False           # also greedy-decode in E1-E6 (slower); teacher-forced metrics are always computed
KEEP_GRID = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # E0/E4 budgets: fraction of fp16 bytes (= tokens kept by selection)
PROBE_BUDGETS = [0.6, 0.8]     # budgets for E2, E3, E6; E0 replaces them when AUTO_PROBE_BUDGETS is True
AUTO_PROBE_BUDGETS = True

if SMOKE_TEST:
    MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT = ["mmlu_abstract_algebra"], 4, 0
    LONGBENCH_TASKS, LONGBENCH_LIMIT = ["longbench_hotpotqa"], 2
    MAX_INPUT_TOKENS = 2048
    RETRIEVAL_DOCS, RETRIEVAL_FACTS, RETRIEVAL_TOKENS = 1, 6, 800
    BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS = 1, 512, 8
    BUDGETS = [0.2, "int4"]
    LAMBDA_SWEEP = [0.0, 1.0]
    EXP_DOCS, KEEP_GRID, PROBE_BUDGETS = 1, [0.5], [0.5]

RESULTS_DIR = Path("results")
set_seed(42)


## 3. Attention backend: recording $a_{i,l}$
`kv_compression_attention` replaces eager attention. While `RECORDER` is active (only during the prefill
of a compressed method), each layer stores
$a_{i,l} = \operatorname{mean}_{h,\,q \in \text{window}} \operatorname{softmax}(q K^\top/\sqrt{d})_i$.


In [ ]:
import math
import torch.nn.functional as F
from transformers import AttentionInterface


class AttentionScoreRecorder:
    """While active, stores a_{i,l} for every layer l: attention from the last `window`
    query positions to the first `n_keys` keys (default: all), reduced over query heads
    and window queries.

    reduce: "mean"          average over heads and window queries (SnapKV-style, the default)
            "mean_visible"  like "mean", but each key is averaged only over the window queries
                            that can see it (needed when the window covers the whole prefix)
            "max"           strongest single (head, query) pair (KVzip-style)"""

    def __init__(self):
        self.active, self.window, self.n_keys, self.reduce, self.scores = False, 32, None, "mean", {}

    def start(self, window, n_keys=None, reduce="mean"):
        self.active, self.window, self.n_keys, self.reduce, self.scores = True, window, n_keys, reduce, {}

    def stop(self):
        scores, self.active, self.scores = self.scores, False, {}
        return scores


RECORDER = AttentionScoreRecorder()
SCORE_QUERY_CHUNK = 128  # window queries scored at a time; bounds memory for long scoring windows


def _window_attention(query, key, scaling, window, n_keys=None, reduce="mean"):
    """[b, Hq, q, d] x [b, Hkv, k, d] -> [b, n_keys] attention mass from the last `window` queries.
    Grouped-query heads are handled without materialising repeated keys, and the window is
    processed in chunks of SCORE_QUERY_CHUNK queries."""
    b, hq, q_len, d = query.shape
    hk, k_len = key.shape[1], key.shape[2]
    n_rep, w = hq // hk, min(window, q_len)
    n_keys = k_len if n_keys is None else min(n_keys, k_len)
    key_t = key.float().transpose(-1, -2)
    key_pos = torch.arange(k_len, device=query.device)
    out = torch.zeros(b, n_keys, device=query.device)
    for start in range(q_len - w, q_len, SCORE_QUERY_CHUNK):
        stop = min(start + SCORE_QUERY_CHUNK, q_len)
        q = query[:, :, start:stop].float().reshape(b, hk, n_rep * (stop - start), d)  # rows ordered (rep, query pos)
        logits = torch.matmul(q, key_t) * scaling
        q_pos = (torch.arange(start, stop, device=query.device) + (k_len - q_len)).repeat(n_rep)
        logits.masked_fill_(key_pos[None, :] > q_pos[:, None], float("-inf"))
        probs = logits.softmax(-1)[..., :n_keys]
        if reduce == "max":
            out = torch.maximum(out, probs.amax(dim=(1, 2)))
        else:
            out += probs.sum(dim=(1, 2))
    if reduce == "max":
        return out
    if reduce == "mean_visible":  # number of window queries at or after each key position
        visible = (k_len - key_pos[:n_keys].clamp_min(k_len - w)).clamp_min(1)
        return out / (hk * n_rep * visible)
    return out / (hk * n_rep * w)


def kv_compression_attention(module, query, key, value, attention_mask, scaling=None, dropout=0.0, **kwargs):
    """SDPA attention that (1) optionally records window attention scores and
    (2) builds its own mask, so it works on caches whose length no longer equals the
    position index (after eviction). Assumes unpadded batches (no left padding)."""
    scaling = scaling if scaling is not None else 1.0 / math.sqrt(query.shape[-1])
    if RECORDER.active:
        RECORDER.scores[module.layer_idx] = _window_attention(query, key, scaling, RECORDER.window,
                                                              RECORDER.n_keys, RECORDER.reduce)
    n_rep = query.shape[1] // key.shape[1]
    if n_rep > 1:
        key = key.repeat_interleave(n_rep, dim=1)
        value = value.repeat_interleave(n_rep, dim=1)
    q_len, k_len = query.shape[2], key.shape[2]
    if q_len == 1:  # decoding: every retained cache entry is visible
        out = F.scaled_dot_product_attention(query, key, value, scale=scaling)
    elif q_len == k_len:  # prefill with an empty cache
        out = F.scaled_dot_product_attention(query, key, value, is_causal=True, scale=scaling)
    else:  # several new tokens on top of a (compressed) cache: causal among the new tokens
        q_pos = torch.arange(k_len - q_len, k_len, device=query.device)
        allowed = torch.arange(k_len, device=query.device)[None, :] <= q_pos[:, None]
        out = F.scaled_dot_product_attention(query, key, value, attn_mask=allowed, scale=scaling)
    return out.transpose(1, 2).contiguous(), None


AttentionInterface.register("kv_compress", kv_compression_attention)
print("Registered attention backend 'kv_compress'.")


## 4. ILF scoring and bit allocation
For every layer $l$ and candidate token $i$:

$$d_i=\sum_{l=1}^{L}\mathbf 1[a_{i,l}>\tau_l],\qquad \mathrm{ILF}_i=\log\frac{L+1}{d_i+1},\qquad S_{i,l}=a_{i,l}\,(1+\lambda\,\mathrm{ILF}_i)$$

By default $\tau_l$ is layer $l$'s $(1-k)$ quantile of attention, where $k$ is the fraction of tokens the
budget lets us keep. So $d_i$ counts the layers in which plain attention selection would have kept
token $i$. Tokens that every layer already wants get a small ILF boost; tokens that only a few layers
find important get a larger one.

* **select**: per layer, the top tokens by $S_{i,l}$ are kept at 16 bits; the rest are evicted.
* **mixed**: per layer, tokens ranked by $S_{i,l}$ fill a 16-bit tier, then an 8-bit tier, then a 4-bit
  tier; the rest get 0 bits (evicted). `TIER_SPLIT` sets how the byte budget is shared across tiers.
* $\lambda = 0$ recovers the plain attention-based baselines.


In [ ]:
from dataclasses import dataclass
from typing import Optional, Union


@dataclass(frozen=True)
class KVCompressionConfig:
    """One compression method.

    method:  "full"   - no compression (baseline)
             "select" - dynamic selection: keep the top tokens at 16 bits, evict the rest
             "quant"  - uniform quantization of every token to `quant_bits`
             "mixed"  - mixed precision: tokens ranked by score get 16 / 8 / 4 / 0 bits
    budget:  fraction of the fp16 KV-cache bytes to keep, or "int4"/"int8" to match the
             byte footprint of uniform 4/8-bit quantization on the same input.
    lam:     lambda in S_{i,l} = a_{i,l} (1 + lambda * ILF_i). lam = 0 is plain attention scoring.
    score_mode: how per-layer attention a_{i,l} becomes the selection score (see `selection_scores`):
             "ilf" (default), "ilf_gated", "lf", "mix", "segment", "idf", "random_shared", "random_layer".
    """

    name: str
    method: str = "full"
    budget: Union[float, str] = 1.0
    lam: float = 0.0
    quant_bits: int = 8
    tier_split: tuple = (1 / 3, 1 / 3, 1 / 3)  # share of the budget spent on 16 / 8 / 4-bit tiers
    sink_tokens: int = 4    # first tokens, always kept at 16 bits (attention sinks)
    window: int = 32        # last tokens, always kept at 16 bits; their queries score the rest
    pool_kernel: int = 7    # 1-D average pooling of scores along tokens (SnapKV-style); 1 disables
    tau_quantile: Optional[float] = None  # per-layer tau_l quantile; None -> 1 - keep fraction
    score_mode: str = "ilf"
    layer_mix: float = 0.0  # "mix": 0 = every layer ranks on its own, 1 = one shared ranking for all layers
    n_segments: int = 16    # "segment": number of contiguous prefix blocks that get equal keep quotas
    seed: int = 0           # "random_*" modes


def bytes_ratio(bits, head_dim, group=32):
    """Approximate bytes per stored token relative to fp16, including the fp16 scale + zero
    stored for every group of `group` elements (keys and values alike)."""
    if bits == 16:
        return 1.0
    if bits == 0:
        return 0.0
    return (bits / 8 + 4 / min(group, head_dim)) / 2


def resolve_budget(cfg, n_tokens, n_forced, head_dim):
    if isinstance(cfg.budget, str):  # "int4" / "int8": match uniform quantization
        bits = int(cfg.budget.removeprefix("int"))
        return (n_forced + (n_tokens - n_forced) * bytes_ratio(bits, head_dim)) / n_tokens
    return float(cfg.budget)


def tier_counts(cfg, n_tokens, n_forced, head_dim):
    """Number of candidate tokens (outside sinks/window) to store at 16, 8 and 4 bits."""
    n_cand = n_tokens - n_forced
    budget = resolve_budget(cfg, n_tokens, n_forced, head_dim)
    remaining = max(budget * n_tokens - n_forced, 0.0)  # in fp16-token units
    if cfg.method == "select":
        return min(n_cand, int(remaining)), 0, 0
    counts, left = [], n_cand
    for share, bits in zip(cfg.tier_split, (16, 8, 4)):
        n = min(left, int(remaining * share / bytes_ratio(bits, head_dim)))
        counts.append(n)
        left -= n
    return tuple(counts)


def ilf_diversified_scores(attn, lam, keep_fraction, tau_quantile=None):
    """attn: a_{i,l} as [L, batch, tokens].
    d_i     = sum_l 1[a_{i,l} > tau_l]
    ILF_i   = log((L + 1) / (d_i + 1))
    S_{i,l} = a_{i,l} (1 + lam * ILF_i)
    tau_l defaults to layer l's (1 - keep_fraction) quantile, i.e. "token i would be kept
    by plain attention selection in layer l"."""
    n_layers = attn.shape[0]
    q = tau_quantile if tau_quantile is not None else 1.0 - keep_fraction
    q = min(max(q, 0.0), 1.0)
    tau = torch.quantile(attn, q, dim=-1, keepdim=True)
    d = (attn > tau).sum(0)
    ilf = torch.log((n_layers + 1) / (d.float() + 1))
    return attn * (1 + lam * ilf), d, ilf


def _rank01(x):
    """Rank of every entry along the last dim, scaled to [0, 1] (1 = largest)."""
    return x.argsort(-1).argsort(-1).float() / max(x.shape[-1] - 1, 1)


def selection_scores(attn, cfg, keep_fraction, token_ids=None):
    """Selection score S_{i,l} [L, batch, tokens] from attention a_{i,l}, by `cfg.score_mode`.

    "ilf"            a (1 + lam * ILF_i): the proposal. More cross-layer diversity as lam grows.
    "ilf_gated"      as "ilf", but tokens that no layer would keep (d_i = 0) get no boost.
    "lf"             a (1 + lam * log(1 + d_i)): the opposite of ILF, boosts tokens many layers want.
    "mix"            (1 - layer_mix) * r_{i,l} + layer_mix * mean_l r_{i,l}, r = per-layer rank in [0, 1].
                     layer_mix = 0 is plain attention; layer_mix = 1 keeps the same tokens in every layer.
    "segment"        rank of a_{i,l} inside its block of the prefix (`n_segments` equal blocks),
                     so every block keeps the same share of its tokens (diversity over positions).
    "idf"            a (1 + lam * log(M / count_i)), count_i = occurrences of token i's id among the
                     M candidates (diversity over content: literal tf-idf). Needs `token_ids` [batch, tokens].
    "random_shared"  random ranking shared by all layers.   "random_layer": independent per layer."""
    mode, n_tok = cfg.score_mode, attn.shape[-1]
    if mode == "ilf":
        return ilf_diversified_scores(attn, cfg.lam, keep_fraction, cfg.tau_quantile)[0]
    if mode in ("ilf_gated", "lf"):
        _, d, ilf = ilf_diversified_scores(attn, cfg.lam, keep_fraction, cfg.tau_quantile)
        boost = ilf * (d > 0) if mode == "ilf_gated" else torch.log1p(d.float())
        return attn * (1 + cfg.lam * boost)
    if mode == "mix":
        r = _rank01(attn)
        return (1 - cfg.layer_mix) * r + cfg.layer_mix * r.mean(0, keepdim=True)
    if mode == "segment":
        s = torch.empty_like(attn, dtype=torch.float32)
        edges = torch.linspace(0, n_tok, min(cfg.n_segments, n_tok) + 1).round().long().tolist()
        for lo, hi in zip(edges[:-1], edges[1:]):
            s[..., lo:hi] = _rank01(attn[..., lo:hi])
        return s
    if mode == "idf":
        counts = (token_ids[:, :, None] == token_ids[:, None, :]).sum(-1).float()  # [batch, tokens]
        return attn * (1 + cfg.lam * torch.log(n_tok / counts))
    if mode in ("random_shared", "random_layer"):
        g = torch.Generator().manual_seed(cfg.seed)
        shape = attn.shape if mode == "random_layer" else (1, *attn.shape[1:])
        return torch.rand(shape, generator=g).to(attn.device).expand(attn.shape)
    raise ValueError(f"unknown score_mode {mode!r}")


def allocate_bits(scores, n16, n8, n4):
    """scores [batch, tokens] -> bits [batch, tokens]: best n16 -> 16, next n8 -> 8, next n4 -> 4, rest 0."""
    b, m = scores.shape
    tiers = torch.zeros(m, dtype=torch.int16, device=scores.device)
    tiers[:n16] = 16
    tiers[n16:n16 + n8] = 8
    tiers[n16 + n8:n16 + n8 + n4] = 4
    order = scores.argsort(dim=-1, descending=True)
    return torch.zeros_like(order, dtype=torch.int16).scatter_(-1, order, tiers.expand(b, m).contiguous())


def make_config(name, method, budget=1.0, lam=0.0, **kw):
    defaults = dict(tier_split=TIER_SPLIT, sink_tokens=SINK_TOKENS, window=OBS_WINDOW, pool_kernel=POOL_KERNEL)
    return KVCompressionConfig(name, method, budget, lam, **{**defaults, **kw})


def budget_tag(b):
    return b if isinstance(b, str) else f"{b:.0%}"


## 5. Compressed cache with packed mixed-precision storage
`CompressedLayer` plugs into the Transformers `DynamicCache`. It holds a 16-bit block plus packed
8-bit and 4-bit blocks (with metadata) for retained tokens; evicted tokens are simply absent.
`CompressedPrefix.make_cache()` builds a fresh cache that shares the stored tensors, so one compressed
prefix can serve many queries without copying.


In [ ]:
from transformers.cache_utils import DynamicCache, DynamicLayer


QUANT_GROUP = 32  # keys: per-channel over groups of 32 tokens (KIVI-style); values: per-token, 32 channels


@dataclass
class QTensor:
    """Packed asymmetric min-max quantization with fp16 scale/zero per group.
    axis="channel": each channel is quantized over a group of `group` tokens (used for keys,
    which have outlier channels); axis="token": each token over `group` channels (values).
    4-bit codes are packed two per uint8 byte, so storage really shrinks."""

    data: torch.Tensor   # uint8 [b, h, n_padded, d] (or d/2 when 4-bit)
    scale: torch.Tensor
    zero: torch.Tensor
    bits: int
    axis: str
    group: int
    num_tokens: int

    @property
    def nbytes(self):
        return sum(t.numel() * t.element_size() for t in (self.data, self.scale, self.zero))

    def dequantize(self, dtype):
        q = self.data
        if self.bits == 4:
            q = torch.stack((q & 0xF, q >> 4), dim=-1).flatten(-2)
        elif self.bits == 2:
            q = torch.stack((q & 3, (q >> 2) & 3, (q >> 4) & 3, q >> 6), dim=-1).flatten(-2)
        b, h, n, d = q.shape
        shape = (b, h, n // self.group, self.group, d) if self.axis == "channel" else (b, h, n, d // self.group, self.group)
        x = q.to(dtype).reshape(shape) * self.scale.to(dtype) + self.zero.to(dtype)
        return x.reshape(b, h, n, d)[:, :, :self.num_tokens]


def quantize(x, bits, axis, group=QUANT_GROUP):
    assert bits in (2, 4, 8)
    b, h, n, d = x.shape
    xf = x.float()
    if axis == "channel":
        pad = (-n) % group  # pad by repeating the last token so it does not widen the range
        xf = torch.cat([xf, xf[:, :, -1:].expand(b, h, pad, d)], dim=2)
        xg, red = xf.reshape(b, h, (n + pad) // group, group, d), -2
    else:
        group = min(group, d)
        xg, red = xf.reshape(b, h, n, d // group, group), -1
    lo, hi = xg.amin(red, keepdim=True), xg.amax(red, keepdim=True)
    qmax = 2 ** bits - 1
    scale = ((hi - lo) / qmax).clamp_min(1e-5)
    q = torch.round((xg - lo) / scale).clamp_(0, qmax).to(torch.uint8).reshape(b, h, -1, d)
    if bits == 4:
        q = q[..., 0::2] | (q[..., 1::2] << 4)
    elif bits == 2:  # four codes per byte (head_dim must be a multiple of 4)
        q = q[..., 0::4] | (q[..., 1::4] << 2) | (q[..., 2::4] << 4) | (q[..., 3::4] << 6)
    return QTensor(q.contiguous(), scale.half(), lo.half(), bits, axis, group, n)


class CompressedLayer(DynamicLayer):
    """One layer's retained KV entries: a 16-bit part plus packed 8/4-bit parts.
    Newly generated tokens are appended to the 16-bit part. Keys are stored post-RoPE,
    so evicted/reordered entries keep their original positions."""

    def __init__(self, keys16, values16, quantized=()):
        super().__init__()
        self.dtype, self.device = keys16.dtype, keys16.device
        self.keys, self.values = keys16, values16
        self.quantized = list(quantized)  # [(QTensor keys, QTensor values), ...]
        self.is_initialized = True

    def update(self, key_states, value_states, cache_kwargs=None):
        self.keys = torch.cat([self.keys, key_states], dim=-2)
        self.values = torch.cat([self.values, value_states], dim=-2)
        if not self.quantized:
            return self.keys, self.values
        keys = [qk.dequantize(self.dtype) for qk, _ in self.quantized] + [self.keys]
        values = [qv.dequantize(self.dtype) for _, qv in self.quantized] + [self.values]
        return torch.cat(keys, dim=-2), torch.cat(values, dim=-2)

    def get_seq_length(self):
        return self.keys.shape[-2] + sum(qk.num_tokens for qk, _ in self.quantized)


def storage_bytes(layers):
    """KV bytes with 16-bit entries counted at 2 bytes (independent of compute dtype)."""
    total = 0
    for k, v, quantized in layers:
        total += 2 * (k.numel() + v.numel())
        total += sum(qk.nbytes + qv.nbytes for qk, qv in quantized)
    return total


@dataclass
class CompressedPrefix:
    """A compressed, reusable prefix cache. `make_cache()` returns a fresh cache for each
    query, so one compressed prefix can serve many queries (prefix caching)."""

    layers: list   # per layer: (keys16, values16, [(QTensor, QTensor), ...])
    length: int    # number of original prefix tokens = next position id
    stats: dict
    bits: Optional[torch.Tensor] = None  # the bit plan [L, batch, N]: 16 / 8 / 4 / 2 bits, 0 = evicted

    def make_cache(self):
        cache = DynamicCache()
        cache.layers = [CompressedLayer(k, v, q) for k, v, q in self.layers]
        return cache


def _gather_tokens(x, idx):
    return x.gather(2, idx[:, None, :, None].expand(-1, x.shape[1], -1, x.shape[3]))


def _positions_with(bits_l, value):
    """Sorted positions holding `value` bits, [batch, n]; every batch row has the same count."""
    mask = bits_l == value
    n = int(mask[0].sum())
    return mask.nonzero()[:, 1].view(bits_l.shape[0], n) if n else None


def build_layer(keys, values, bits_l):
    idx16 = _positions_with(bits_l, 16)
    k16, v16 = _gather_tokens(keys, idx16), _gather_tokens(values, idx16)
    quantized = []
    for b in (8, 4, 2):
        idx = _positions_with(bits_l, b)
        if idx is not None:
            quantized.append((quantize(_gather_tokens(keys, idx), b, "channel"),
                              quantize(_gather_tokens(values, idx), b, "token")))
    return k16, v16, quantized


## 6. Prefill → compress → continue
`prefill_and_compress` returns the compressed prefix and its statistics: bytes kept, the share of tokens at
each bit width, unique positions kept across layers, and the mean pairwise Jaccard overlap of
the layers' kept sets. `extend` and `greedy_generate` continue from it at the original positions.


In [ ]:
def model_dims(model):
    c = model.config
    head_dim = getattr(c, "head_dim", None) or c.hidden_size // c.num_attention_heads
    return c.num_hidden_layers, c.num_key_value_heads, head_dim


def selection_stats(bits, sink, window):
    """Diversity diagnostics from the bit plan bits [L, batch, N]."""
    n_layers, _, n_tokens = bits.shape
    cand = bits[..., sink:n_tokens - window] > 0  # candidate positions only (forced ones excluded)
    stats = {f"frac_tokens_{b}bit": (bits == b).float().mean().item() for b in (16, 8, 4, 2, 0)}
    stats["tokens_retained_frac"] = (bits > 0).float().mean().item()
    if cand.shape[-1] == 0:
        return stats
    kept_any = cand.any(0).float()  # [batch, M]
    stats["unique_positions"] = kept_any.sum(-1).mean().item()
    stats["unique_positions_frac"] = kept_any.mean().item()
    stats["mean_kept_per_layer"] = cand.float().sum(-1).mean().item()
    r = cand.float()
    inter = torch.einsum("lbm,kbm->blk", r, r)
    sizes = r.sum(-1).T  # [batch, L]
    union = sizes[:, :, None] + sizes[:, None, :] - inter
    jac = inter / union.clamp_min(1)
    off_diag = ~torch.eye(n_layers, dtype=torch.bool, device=bits.device)
    stats["mean_layer_jaccard"] = jac[:, off_diag].mean().item()
    return stats


@torch.inference_mode()
def prefill_and_compress(model, input_ids, cfg, score_suffix_ids=None, score_window=None, score_reduce="mean",
                         protect=None, bit_override=None):
    """Prefill `input_ids` [batch, N] (unpadded) and compress the resulting KV cache.

    score_suffix_ids  [batch, S] tokens appended for scoring only: a_{i,l} is the attention from
                      these S tokens to the prefix. Their KV entries are dropped before compression
                      (exact, because attention is causal), so the result is still a prefix-only cache.
    score_window      without a suffix: number of trailing prefix tokens whose queries give a_{i,l}
                      (default `cfg.window`).
    score_reduce      "mean" | "mean_visible" | "max", see AttentionScoreRecorder.
    protect           bool [batch, N]: candidate tokens that are ranked first in every layer (oracle).
    bit_override      int [L, batch, N]: entries >= 0 replace the bit plan after allocation (oracle).
    Returns (CompressedPrefix, next-token logits of the prefix, or None when a suffix was used)."""
    batch, n_tokens = input_ids.shape
    n_layers, n_kv_heads, head_dim = model_dims(model)
    need_scores = cfg.method in ("select", "mixed")
    t0 = time.perf_counter()
    cache = DynamicCache()
    run_ids = input_ids if score_suffix_ids is None else torch.cat([input_ids, score_suffix_ids], dim=1)
    if need_scores:
        n_queries = score_suffix_ids.shape[1] if score_suffix_ids is not None else (score_window or cfg.window)
        RECORDER.start(n_queries, n_keys=n_tokens, reduce=score_reduce)
    try:
        out = model(input_ids=run_ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
    finally:
        scores = RECORDER.stop()
    if score_suffix_ids is None:
        last_logits = out.logits[:, -1].float()
        full = [(layer.keys, layer.values) for layer in cache.layers]
    else:  # keep only the prefix part of the cache
        last_logits = None
        full = [(layer.keys[:, :, :n_tokens].clone(), layer.values[:, :, :n_tokens].clone()) for layer in cache.layers]
    del out, cache

    sink = min(cfg.sink_tokens, n_tokens)
    window = min(cfg.window, n_tokens - sink)
    n_forced, n_cand = sink + window, n_tokens - sink - window
    bits = torch.full((n_layers, batch, n_tokens), 16, dtype=torch.int16, device=input_ids.device)
    compress = cfg.method != "full" and n_cand > 0
    if compress:
        if need_scores:
            attn = torch.stack([scores[l] for l in range(n_layers)])  # a_{i,l}: [L, batch, N]
            if cfg.pool_kernel > 1:
                k = cfg.pool_kernel
                attn = F.avg_pool1d(attn.reshape(-1, 1, n_tokens), k, stride=1, padding=k // 2,
                                    count_include_pad=False).reshape(n_layers, batch, n_tokens)
            n16, n8, n4 = tier_counts(cfg, n_tokens, n_forced, head_dim)
            keep_fraction = (n16 + n8 + n4) / n_cand
            cand = slice(sink, n_tokens - window)
            s = selection_scores(attn[..., cand], cfg, keep_fraction, input_ids[:, cand])
            if protect is not None:
                s = s.masked_fill(protect[None, :, cand], float("inf"))
            bits[..., cand] = torch.stack([allocate_bits(s[l], n16, n8, n4) for l in range(n_layers)])
        elif cfg.method == "quant":
            bits[..., sink:n_tokens - window] = cfg.quant_bits
        else:
            raise ValueError(f"unknown method {cfg.method!r}")
    if bit_override is not None:
        bits = torch.where(bit_override >= 0, bit_override.to(bits), bits)
        compress = True
    if compress:
        layers = [build_layer(k, v, bits[l]) for l, (k, v) in enumerate(full)]
    else:
        layers = [(k, v, []) for k, v in full]
    del full
    if input_ids.is_cuda:
        torch.cuda.synchronize()

    full_bytes = n_layers * 2 * batch * n_kv_heads * n_tokens * head_dim * 2
    kv_bytes = storage_bytes(layers)
    stats = {
        "prefix_tokens": n_tokens,
        "kv_bytes": kv_bytes,
        "full_kv_bytes": full_bytes,
        "kv_bytes_pct": 100 * kv_bytes / full_bytes,
        "prefill_compress_s": time.perf_counter() - t0,
        **selection_stats(bits, sink, window),
    }
    return CompressedPrefix(layers, n_tokens, stats, bits), last_logits


@torch.inference_mode()
def extend(model, cache, token_ids, start, all_logits=False):
    """Run new tokens [batch, n] on top of `cache`, at original positions start..start+n-1."""
    n = token_ids.shape[1]
    pos = torch.arange(start, start + n, device=token_ids.device)
    out = model(input_ids=token_ids, past_key_values=cache, use_cache=True,
                position_ids=pos[None].expand(token_ids.shape[0], -1), cache_position=pos,
                logits_to_keep=0 if all_logits else 1)
    return out.logits.float()


@torch.inference_mode()
def score_continuation(model, prefix, suffix_ids, target_ids):
    """Teacher-forced scoring on a compressed prefix, in one forward pass: log-probability of each
    token of `target_ids` [1, m] following `suffix_ids` [1, n], and the greedy prediction at each step."""
    m = target_ids.shape[1]
    feed = torch.cat([suffix_ids, target_ids[:, :-1]], dim=1)
    logits = extend(model, prefix.make_cache(), feed, prefix.length, all_logits=True)[0, -m:]
    logprobs = logits.log_softmax(-1).gather(-1, target_ids[0][:, None])[:, 0]
    return logprobs, logits.argmax(-1)


def eos_ids(model, tokenizer):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, (list, tuple)) else [ids])
    ids.add(tokenizer.eos_token_id)
    return {i for i in ids if i is not None}


@torch.inference_mode()
def greedy_generate(model, tokenizer, prefix, suffix_ids, max_new_tokens, stop=()):
    """Greedy decoding (batch 1) from a compressed prefix followed by `suffix_ids` [1, n]."""
    cache = prefix.make_cache()
    logits = extend(model, cache, suffix_ids, prefix.length)[:, -1]
    pos, out, stop_ids = prefix.length + suffix_ids.shape[1], [], eos_ids(model, tokenizer)
    for _ in range(max_new_tokens):
        nxt = int(logits.argmax(-1))
        if nxt in stop_ids:
            break
        out.append(nxt)
        if stop and any(s in tokenizer.decode(out, skip_special_tokens=True) for s in stop):
            break
        logits = extend(model, cache, torch.tensor([[nxt]], device=suffix_ids.device), pos)[:, -1]
        pos += 1
    text = tokenizer.decode(out, skip_special_tokens=True)
    for s in stop:
        text = text.split(s)[0]
    return text


## 7. Load the model
The model uses the `kv_compress` attention backend for every method, including the uncompressed
baseline, so every comparison runs on the same kernels.


In [ ]:
def load_model(model_id):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
             else torch.float16 if device == "cuda" else torch.float32)
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, attn_implementation="kv_compress")
    mdl = mdl.to(device).eval()
    max_len = min(MAX_INPUT_TOKENS, getattr(mdl.config, "max_position_embeddings", MAX_INPUT_TOKENS))
    print(f"{model_id} | {mdl.device} | {mdl.dtype} | layers/kv-heads/head-dim = {model_dims(mdl)} | max input {max_len}")
    return mdl, tok, max_len


model, tokenizer, MAX_LEN = load_model(MODEL_ID)


### Sanity checks and a look at ILF
1. With no compression, prefill + `extend` must reproduce the logits of one ordinary forward pass.
2. On a short context, compare the tokens kept by plain attention selection and by ILF selection in
   one layer, and check how much ILF raises the number of unique positions kept across layers.


In [ ]:
CONTEXT = (
    "The archive has three rooms. The red room holds maps. The blue room holds a brass key. "
    "The green room holds letters. The archivist, Ms. Okafor, locks the blue room every night at nine. "
    "Visitors must sign the ledger by the door and may not bring food inside. "
) * 3 + "Question: Which room holds the brass key? Answer: The"
ids = tokenizer(CONTEXT, return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    reference = model(ids).logits.float()
cut = ids.shape[1] - 4
prefix, _ = prefill_and_compress(model, ids[:, :cut], make_config("full", "full"))
logits = extend(model, prefix.make_cache(), ids[:, cut:], cut, all_logits=True)
print(f"max |logit difference| full-cache path vs plain forward: {(logits - reference[:, cut:]).abs().max():.2e}")

# ILF on this context, layer by layer
with torch.inference_mode():
    RECORDER.start(OBS_WINDOW)
    model(ids)
    scores = RECORDER.stop()
attn = torch.stack([scores[l] for l in range(len(scores))])[:, 0]  # [L, N]
cand = attn[:, SINK_TOKENS:-OBS_WINDOW]
_, d, ilf = ilf_diversified_scores(cand[:, None], LAMBDA, keep_fraction=0.25)
tokens = tokenizer.convert_ids_to_tokens(ids[0, SINK_TOKENS:-OBS_WINDOW])
print(f"\n{cand.shape[1]} candidate tokens, {cand.shape[0]} layers. Most and least 'common' tokens (d_i):")
order = d[0].argsort(descending=True)
print("  high d_i (low ILF):", [(tokens[i], int(d[0, i])) for i in order[:8]])
print("  low d_i (high ILF):", [(tokens[i], int(d[0, i])) for i in order[-8:]])

rows = []
for name, lam in [("attention (λ=0)", 0.0), (f"ILF (λ={LAMBDA})", LAMBDA)]:
    p, _ = prefill_and_compress(model, ids[:, :cut], make_config(name, "select", 0.35, lam))
    rows.append({"selection": name, **{k: round(p.stats[k], 3) for k in
                 ["kv_bytes_pct", "unique_positions", "unique_positions_frac", "mean_layer_jaccard"]},
                 "answer": greedy_generate(model, tokenizer, p, ids[:, cut:], 6, ["\n"])})
pd.DataFrame(rows)


## 8. Methods and matched budgets
Every row in the grid is a `KVCompressionConfig`. At each budget there are four ablation cells:
{attention, ILF} × {selection only, mixed precision}. They are compared against the full cache and
uniform int8/int4 quantization. The `@int4`/`@int8` budgets are byte-matched to the uniform-quantization rows.


In [ ]:
def method_grid(budgets=BUDGETS, lam=LAMBDA):
    """Ablation grid at matched memory budgets:
    attn-select vs ilf-select      -> effect of ILF diversification on selection
    *-select vs *-mixed            -> effect of mixed precision on top of selection
    quant-int4/8 vs *-mixed@int4/8 -> selection + mixed precision vs uniform quantization"""
    methods = [make_config("full", "full"),
               make_config("quant-int8", "quant", quant_bits=8),
               make_config("quant-int4", "quant", quant_bits=4)]
    for b in budgets:
        t = budget_tag(b)
        methods += [make_config(f"attn-select@{t}", "select", b, 0.0),
                    make_config(f"ilf-select@{t}", "select", b, lam),
                    make_config(f"attn-mixed@{t}", "mixed", b, 0.0),
                    make_config(f"ilf-mixed@{t}", "mixed", b, lam)]
    return methods


def describe(cfg):
    return {"method": cfg.name, "family": cfg.name.split("@")[0], "kind": cfg.method,
            "budget": str(cfg.budget), "lambda": cfg.lam}


def save_table(df, model_id, name):
    path = RESULTS_DIR / model_id.replace("/", "__") / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved {path}")


METHODS = method_grid()
print(f"{len(METHODS)} methods:", [m.name for m in METHODS])


## 9. Evaluation helpers
`CompressedKVLM` is an lm-evaluation-harness model that routes every request through the compressed
cache (details in its docstring). The other helpers run the retrieval and memory/latency experiments.


In [ ]:
from lm_eval import evaluator
from lm_eval.api.model import LM
from lm_eval.utils import make_table
from tqdm.auto import tqdm

# Where the question starts in each LongBench prompt. With QUERY_AWARE = False, everything
# before the marker is the reusable prefix: it is compressed *without seeing the question*,
# then the question is run on top of the compressed cache (prefix-caching setup).
LONGBENCH_QUESTION_MARKERS = [
    "\n\nAnswer the question based on the given passages.",          # hotpotqa, 2wikimqa, musique
    "\n\nNow, answer the following question based on the above text",  # multifieldqa_en
    "\n\n Answer the question based on the above article",            # qasper
    "\n\nNow, answer the question based on the story",                 # narrativeqa
    "\n\nThe following is an abstract.",                               # passage_retrieval_en
    "\n\nNow, write a one-page summary of the report.",                # gov_report
]


def middle_truncate(ids, max_len):
    """LongBench convention: keep the first and last halves of an over-long prompt."""
    if len(ids) <= max_len:
        return ids
    half = max_len // 2
    return ids[:half] + ids[len(ids) - (max_len - half):]


class CompressedKVLM(LM):
    """lm-evaluation-harness model that runs every request through a compressed KV cache.

    loglikelihood (MMLU): the context minus its last token is the prefix; it is compressed,
    then [last context token + continuation] is scored on top of the compressed cache, so
    every scored token depends on the compressed cache. Requests sharing a context reuse
    one compressed prefix.
    generate_until (LongBench): prefix = prompt up to the question (QUERY_AWARE=False) or
    the whole prompt minus its last token (QUERY_AWARE=True); greedy decoding afterwards."""

    def __init__(self, model, tokenizer, cfg, max_input_tokens=4096, query_aware=False,
                 markers=LONGBENCH_QUESTION_MARKERS):
        super().__init__()
        self.model, self.tokenizer, self.cfg = model, tokenizer, cfg
        self.max_input_tokens, self.query_aware, self.markers = max_input_tokens, query_aware, markers
        self.device = model.device
        self.compression_stats = []

    def _ids(self, text, special=True):
        return self.tokenizer(text, add_special_tokens=special).input_ids

    def _tensor(self, ids):
        return torch.tensor([ids], device=self.device)

    def _compress(self, prefix_ids):
        prefix, _ = prefill_and_compress(self.model, self._tensor(prefix_ids), self.cfg)
        self.compression_stats.append(prefix.stats)
        return prefix

    def loglikelihood(self, requests, disable_tqdm=False):
        groups = defaultdict(list)
        for i, req in enumerate(requests):
            context, continuation = req.args
            groups[context].append((i, continuation))
        results = [None] * len(requests)
        for context, items in tqdm(groups.items(), desc=f"loglikelihood [{self.cfg.name}]", disable=disable_tqdm):
            n_space = len(context) - len(context.rstrip())  # move trailing spaces into the continuation
            ctx = context[:len(context) - n_space] if n_space else context
            ctx_ids = self._ids(ctx)
            conts = []
            for i, cont in items:
                cont = context[len(ctx):] + cont
                conts.append((i, self._ids(ctx + cont)[len(ctx_ids):]))
            longest = max(len(c) for _, c in conts)
            ctx_ids = ctx_ids[-(self.max_input_tokens - longest):]
            if len(ctx_ids) < 2:
                raise ValueError("context too short to split into prefix + last token")
            prefix = self._compress(ctx_ids[:-1])
            for i, cont_ids in conts:
                feed = self._tensor([ctx_ids[-1]] + cont_ids[:-1])
                logits = extend(self.model, prefix.make_cache(), feed, prefix.length, all_logits=True)[0]
                logprobs = logits.log_softmax(-1)
                target = torch.tensor(cont_ids, device=logits.device)
                ll = logprobs.gather(-1, target[:, None]).sum().item()
                results[i] = (ll, bool((logits.argmax(-1) == target).all()))
        return results

    def _split(self, context):
        if not self.query_aware:
            cut = max(context.rfind(m) for m in self.markers)
            if cut > 0:
                return context[:cut], context[cut:]
        return None, None

    def generate_until(self, requests, disable_tqdm=False):
        outputs = []
        for req in tqdm(requests, desc=f"generate [{self.cfg.name}]", disable=disable_tqdm):
            context, gen_kwargs = req.args
            until = gen_kwargs.get("until") or []
            until = [until] if isinstance(until, str) else list(until)
            max_new = int(gen_kwargs.get("max_gen_toks", 128))
            budget = self.max_input_tokens - max_new
            prefix_str, suffix_str = self._split(context)
            if prefix_str is None:
                ids = middle_truncate(self._ids(context), budget)
                prefix_ids, suffix_ids = ids[:-1], ids[-1:]
            else:
                suffix_ids = self._ids(suffix_str, special=False)
                prefix_ids = middle_truncate(self._ids(prefix_str), budget - len(suffix_ids))
            prefix = self._compress(prefix_ids)
            outputs.append(greedy_generate(self.model, self.tokenizer, prefix, self._tensor(suffix_ids),
                                           max_new, until))
        return outputs

    def loglikelihood_rolling(self, requests, disable_tqdm=False):
        raise NotImplementedError("perplexity tasks are not used in this project")


def mean_compression_stats(stats_list):
    if not stats_list:
        return {}
    keys = [k for k in stats_list[0] if isinstance(stats_list[0][k], (int, float))]
    return {k: sum(s.get(k, 0) for s in stats_list) / len(stats_list) for k in keys}


def run_lm_eval(model, tokenizer, cfg, tasks, limit, num_fewshot=None, max_input_tokens=4096,
                query_aware=False):
    lm = CompressedKVLM(model, tokenizer, cfg, max_input_tokens, query_aware)
    results = evaluator.simple_evaluate(
        model=lm, tasks=tasks, num_fewshot=num_fewshot, limit=limit, log_samples=False,
        random_seed=42, numpy_random_seed=42, torch_random_seed=42, fewshot_random_seed=42,
    )
    return results, mean_compression_stats(lm.compression_stats)


In [ ]:
import numpy as np

STAT_COLUMNS = ["kv_bytes_pct", "tokens_retained_frac", "unique_positions", "unique_positions_frac",
                "mean_layer_jaccard", "frac_tokens_16bit", "frac_tokens_8bit", "frac_tokens_4bit"]


def headline_metric(task_results):
    """Primary metric of one lm-eval task result dict (accuracy for MMLU, score for LongBench)."""
    for key in ("acc,none", "score,none", "exact_match,none"):
        if key in task_results:
            return key.split(",")[0], task_results[key]
    raise KeyError(f"no known metric in {list(task_results)}")


def run_benchmark_suite(model, tokenizer, methods, tasks, limit, num_fewshot=None, max_input_tokens=4096):
    rows = []
    for cfg in methods:
        results, stats = run_lm_eval(model, tokenizer, cfg, tasks, limit, num_fewshot,
                                     max_input_tokens, QUERY_AWARE)
        for task in tasks:
            metric, value = headline_metric(results["results"][task])
            rows.append({**describe(cfg), "task": task, "metric": metric, "value": value,
                         **{k: stats.get(k) for k in STAT_COLUMNS}})
        print(f"{cfg.name:>22}: " + ", ".join(f"{r['task']}={r['value']:.3f}" for r in rows[-len(tasks):]))
    return pd.DataFrame(rows)


# ---- Multi-query reuse retrieval: one compressed prefix, many questions about distant facts ----
NAMES = ["Alice", "Bruno", "Chiara", "Dmitri", "Esme", "Farid", "Greta", "Hiro", "Ines", "Jonas",
         "Kemal", "Lena", "Mateo", "Nadia", "Oskar", "Priya", "Quentin", "Rosa", "Sven", "Tariq",
         "Uma", "Viktor", "Wen", "Ximena", "Yusuf", "Zara", "Anouk", "Bodhi", "Celine", "Darius"]
FILLER = [
    "The morning shift checked the storage racks and found nothing unusual.",
    "A delivery truck arrived late because of heavy traffic near the bridge.",
    "Someone left a stack of old newspapers beside the photocopier again.",
    "The cafeteria served lentil soup, fresh bread and a small green salad.",
    "Maintenance replaced two flickering lights in the east corridor.",
    "The weekly meeting ran long while the team discussed the new schedule.",
    "Rain kept most visitors away from the courtyard during the afternoon.",
    "An inventory count showed the paper supply was slightly below target.",
    "The fire drill finished in under four minutes, a new building record.",
    "Several plants in the lobby were moved closer to the tall windows.",
    "The printer on the third floor jammed twice before lunch.",
    "A new coffee machine was installed and immediately drew a small crowd.",
]


def make_retrieval_doc(n_facts, n_sentences, seed, sentences=None):
    """A document hiding `n_facts` "secret code" sentences at evenly spaced depths. The haystack is
    `n_sentences` draws (with repetition) from FILLER, or the given list of `sentences`. Names and
    codes depend only on `seed`, so different haystacks hide the same facts."""
    rng = random.Random(seed)
    names = rng.sample(NAMES, n_facts)
    codes = [rng.randint(10000, 99999) for _ in names]
    if sentences is None:
        sentences = [rng.choice(FILLER) for _ in range(n_sentences)]
    else:
        sentences, n_sentences = list(sentences), len(sentences)
    slots = [int((j + 0.5) * n_sentences / n_facts) for j in range(n_facts)]
    for j in reversed(range(n_facts)):
        sentences.insert(slots[j], f"The secret code of {names[j]} is {codes[j]}.")
    doc = ("Below is a long building log. Secret codes for several people are hidden in it. "
           "Remember every one of them.\n\n" + " ".join(sentences) + "\n\n")
    facts = []
    for n, c, s in zip(names, codes, slots):
        sentence = f"The secret code of {n} is {c}."
        start = doc.index(sentence)
        end = start + len(sentence)
        facts.append({"name": n, "code": c, "depth": s / n_sentences,
                      "char_span": (start, end), "code_span": (end - 1 - len(str(c)), end - 1)})
    return doc, facts


_NATURAL_POOL = None


def natural_sentences(min_words=8, max_words=40):
    """Digit-free sentences from the WikiText test split, lightly detokenized (cached).
    No digits, so the only numbers in a natural haystack are the secret codes."""
    global _NATURAL_POOL
    if _NATURAL_POOL is None:
        from datasets import load_dataset
        pool = []
        # WikiText-2 and WikiText-103 share this test split; the "2" config avoids the large train download.
        for line in load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"]:
            line = line.strip()
            if not line or line.startswith("="):
                continue
            line = re.sub(r" @([-,.])@ ", r"\1", line)
            line = re.sub(r" ([.,;:!?%)\]])", r"\1", line)
            line = re.sub(r"([(\[]) ", r"\1", line)
            line = re.sub(r" (n't|'s|'re|'ve|'ll|'d|'m)\b", r"\1", line)
            for s in re.split(r"(?<=[.!?])\s+", line):
                if (min_words <= len(s.split()) <= max_words and s[-1] in ".!?"
                        and not re.search(r"\d", s) and '"' not in s):
                    pool.append(s)
        _NATURAL_POOL = list(dict.fromkeys(pool))
        assert len(_NATURAL_POOL) > 500, "too few natural sentences"
    return _NATURAL_POOL


def retrieval_docs(tokenizer, n_docs, n_facts, n_tokens, haystack="repeat"):
    """haystack="repeat":  the original log, FILLER sentences drawn with repetition.
    haystack="natural": non-repeating Wikipedia prose with the same facts at the same depths."""
    if haystack == "repeat":
        per_sentence = len(tokenizer(" ".join(FILLER), add_special_tokens=False).input_ids) / len(FILLER)
        return [make_retrieval_doc(n_facts, int(n_tokens / per_sentence), seed) for seed in range(n_docs)]
    pool = natural_sentences()
    lengths = [len(t) for t in tokenizer(pool, add_special_tokens=False).input_ids]
    docs = []
    for seed in range(n_docs):
        chosen, total = [], 0
        for i in random.Random(10_000 + seed).sample(range(len(pool)), len(pool)):
            if total + lengths[i] > n_tokens:
                break
            chosen.append(pool[i])
            total += lengths[i]
        docs.append(make_retrieval_doc(n_facts, len(chosen), seed, sentences=chosen))
    return docs


# Scoring-only suffixes (see Arm.scoring). They are never part of the compressed cache.
PROBE_TEXT = "\n\nBefore continuing, list every name, number and key fact stated in the text above:\n"
RECON_TEXT = "\n\nRepeat the text above exactly.\n\n"


@dataclass
class Arm:
    """One experimental arm: a compression config plus where its attention scores come from.

    scoring: "window"       the last `cfg.window` prefix tokens (default; no question in view)
             "all_queries"  attention each token receives from every later prefix token (H2O-style)
             "probe"        PROBE_TEXT appended for scoring only (generic, task-agnostic instruction)
             "reconstruct"  RECON_TEXT + the document again, max over queries (simplified KVzip-style)
             "query"        a question appended for scoring only. The cache is then used for that
                            question (rows with query_match = True) and for other questions (False).
    plan:    optional callable(ctx) -> {"protect": ..., "bit_override": ...} for oracle experiments.
    tags:    extra columns copied into every result row."""

    cfg: KVCompressionConfig
    scoring: str = "window"
    plan: Optional[object] = None
    tags: Optional[dict] = None


def question_text(name):
    return f"Question: What is the secret code of {name}?\nAnswer: The secret code of {name} is"


def encode_doc(tokenizer, doc, facts, device):
    """Token ids [1, N] and, per fact, the token positions of (its whole sentence, its code)."""
    enc = tokenizer(doc, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    spans = []
    for f in facts:
        (s0, s1), (c0, c1) = f["char_span"], f["code_span"]
        spans.append(([i for i, (a, b) in enumerate(offsets) if a < s1 and b > s0],
                      [i for i, (a, b) in enumerate(offsets) if a < c1 and b > c0]))
    return torch.tensor([enc["input_ids"]], device=device), spans


def question_ids(tokenizer, fact, device):
    """Question ids [1, n], gold answer ids [1, m] and a mask of the answer tokens that carry digits."""
    q = question_text(fact["name"])
    q_ids = tokenizer(q, add_special_tokens=False).input_ids
    both = tokenizer(f"{q} {fact['code']}", add_special_tokens=False).input_ids
    gold = both[len(q_ids):] if both[:len(q_ids)] == q_ids else tokenizer(f" {fact['code']}", add_special_tokens=False).input_ids
    is_digit = [any(ch.isdigit() for ch in tokenizer.decode([t])) for t in gold]
    return (torch.tensor([q_ids], device=device), torch.tensor([gold], device=device),
            torch.tensor(is_digit, device=device))


def eval_fact(model, tokenizer, prefix, fact, generate):
    """Ask one question on a compressed prefix.
    Teacher-forced (always): gold_nll = mean negative log-likelihood per gold digit token (nats),
    tf_digit_acc / tf_exact = greedy prediction right at each / every digit token given the gold
    digits before it. With `generate`: the original free-running exact match and digit accuracy."""
    q_ids, gold, is_digit = question_ids(tokenizer, fact, model.device)
    logprobs, pred = score_continuation(model, prefix, q_ids, gold)
    hit = (pred == gold[0])[is_digit]
    row = {"gold_nll": -logprobs[is_digit].mean().item(), "tf_digit_acc": hit.float().mean().item(),
           "tf_exact": bool(hit.all())}
    if generate:
        answer = greedy_generate(model, tokenizer, prefix, q_ids, 8, ["\n"])
        digits = re.sub(r"\D", "", answer)[:5]
        row["correct"] = digits == str(fact["code"])
        # partial credit: fraction of the 5 digits right, in position
        row["digit_acc"] = sum(a == b for a, b in zip(digits, str(fact["code"]))) / 5
    return row


def needle_stats(bits, sentence_tokens, code_tokens):
    """How much of one fact survived compression, from the bit plan [L, 1, N]."""
    kept = bits[:, 0] > 0
    return {"needle_kept_frac": kept[:, sentence_tokens].float().mean().item(),  # over (layer, token) pairs
            "code_layers_full": int(kept[:, code_tokens].all(dim=1).sum())}       # layers holding the whole code


def retention_by_depth(bits, cfg, n_bins=10):
    """Share of candidate tokens kept, per layer and per depth bin of the prefix: [L, n_bins]."""
    n_tokens = bits.shape[-1]
    sink = min(cfg.sink_tokens, n_tokens)
    kept = (bits[:, 0, sink:n_tokens - min(cfg.window, n_tokens - sink)] > 0).float()
    return torch.stack([part.mean(dim=1) for part in torch.tensor_split(kept, n_bins, dim=1)], dim=1).cpu()


def scoring_kwargs(scoring, tokenizer, ids):
    """Arguments for `prefill_and_compress` that implement one Arm.scoring protocol (except "query")."""
    encode = lambda text: torch.tensor([tokenizer(text, add_special_tokens=False).input_ids], device=ids.device)
    if scoring == "window":
        return {}
    if scoring == "all_queries":
        return {"score_window": ids.shape[1], "score_reduce": "mean_visible"}
    if scoring == "probe":
        return {"score_suffix_ids": encode(PROBE_TEXT)}
    if scoring == "reconstruct":
        body = ids[:, 1:] if ids[0, 0].item() == tokenizer.bos_token_id else ids
        return {"score_suffix_ids": torch.cat([encode(RECON_TEXT), body], dim=1), "score_reduce": "max"}
    raise ValueError(f"unknown scoring {scoring!r}")


def query_eval_set(j0, n_facts, offsets=(3, 7, 11, 15)):
    """Facts asked on a cache that was scored with question j0: j0 itself, then up to four others."""
    others = []
    for o in offsets:
        j = (j0 + o) % n_facts
        if j != j0 and j not in others:
            others.append(j)
    return [j0] + others


def run_arms(model, tokenizer, arms, docs, generate=False, retention=None):
    """Run every arm on every document: compress the document, then ask about its facts on the
    compressed cache. Returns one row per (arm, document, question). `retention`, if a dict, is
    filled with per-arm lists of `retention_by_depth` maps (one per document)."""
    n_layers, rows = model_dims(model)[0], []
    for arm in arms:
        cfg, first = arm.cfg, len(rows)
        for d, (doc, facts) in enumerate(docs):
            ids, spans = encode_doc(tokenizer, doc, facts, model.device)
            plan = {} if arm.plan is None else arm.plan(
                {"doc": d, "n_tokens": ids.shape[1], "n_layers": n_layers, "device": ids.device,
                 "fact_tokens": [sentence for sentence, _ in spans]})
            if arm.scoring == "query":  # every second fact supplies the scoring question
                jobs = [({"score_suffix_ids": question_ids(tokenizer, facts[j0], ids.device)[0]},
                         [(j, j == j0) for j in query_eval_set(j0, len(facts))])
                        for j0 in range(0, len(facts), 2)]
            else:
                jobs = [(scoring_kwargs(arm.scoring, tokenizer, ids), [(j, None) for j in range(len(facts))])]
            for score_kw, targets in jobs:
                prefix, _ = prefill_and_compress(model, ids, cfg, **score_kw, **plan)
                if retention is not None:
                    retention.setdefault(cfg.name, []).append(retention_by_depth(prefix.bits, cfg))
                for j, match in targets:
                    rows.append({**describe(cfg), **(arm.tags or {}), "scoring": arm.scoring, "query_match": match,
                                 "doc": d, "fact": j, "depth": facts[j]["depth"],
                                 **eval_fact(model, tokenizer, prefix, facts[j], generate),
                                 **needle_stats(prefix.bits, *spans[j]),
                                 **{k: prefix.stats.get(k) for k in STAT_COLUMNS}})
                del prefix
        mine = pd.DataFrame(rows[first:])
        print(f"{cfg.name:>26} [{arm.scoring}]: tf-exact {mine['tf_exact'].mean():.3f}  "
              f"gold NLL {mine['gold_nll'].mean():.2f}  needle kept {mine['needle_kept_frac'].mean():.2f}  "
              f"KV bytes {mine['kv_bytes_pct'].mean():.1f}%"
              + (f"  exact {mine['correct'].mean():.3f}" if generate else ""))
    return pd.DataFrame(rows)


def run_retrieval(model, tokenizer, methods, docs):
    """Compress each document ONCE, without any question in view, then ask about every fact."""
    return run_arms(model, tokenizer, [Arm(cfg) for cfg in methods], docs, generate=True)


def summarize_arms(df, by=(), keys=None):
    """Mean of every metric per arm (plus the extra `by` columns), or per the given `keys`."""
    keys = list(keys) if keys is not None else ["method", "family", "kind", "budget", "lambda", "scoring", *by]
    metrics = [c for c in ["correct", "digit_acc", "tf_exact", "tf_digit_acc", "gold_nll", "needle_kept_frac",
                           "code_layers_full", *STAT_COLUMNS] if c in df and c not in keys]
    return (df.assign(**{c: df[c].astype(float) for c in ("correct", "tf_exact") if c in df})
              .groupby(keys, sort=False, dropna=False)[metrics].mean().reset_index())


def paired_compare(df, a, b, metric="tf_exact", n_boot=2000, seed=0):
    """Method `b` minus method `a` on the same (document, question) pairs: mean difference with a
    95% bootstrap interval (documents resampled as clusters) and the counts of questions on which
    only one of the two is better. Use on arms with one row per (doc, fact)."""
    x = df[df["method"] == a].set_index(["doc", "fact"])[metric].astype(float)
    y = df[df["method"] == b].set_index(["doc", "fact"])[metric].astype(float)
    diff = (y - x).dropna()
    per_doc = diff.groupby(level="doc").agg(["sum", "count"])
    pick = np.random.default_rng(seed).integers(0, len(per_doc), size=(n_boot, len(per_doc)))
    boots = per_doc["sum"].to_numpy()[pick].sum(1) / per_doc["count"].to_numpy()[pick].sum(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"a": a, "b": b, "metric": metric, "n": len(diff), "mean_a": x.mean(), "mean_b": y.mean(),
            "diff_b_minus_a": diff.mean(), "ci_lo": lo, "ci_hi": hi,
            "a_better": int((diff < 0).sum()), "b_better": int((diff > 0).sum())}


def summarize_retrieval(df):
    keys = ["method", "family", "kind", "budget", "lambda"]
    return (df.groupby(keys, sort=False)
              .agg(accuracy=("correct", "mean"), digit_acc=("digit_acc", "mean"), **{c: (c, "mean") for c in STAT_COLUMNS})
              .reset_index())


# ---- Memory and decode latency at a fixed workload (batch x sequence length) ----
def benchmark_memory_latency(model, tokenizer, cfg, batch, seq_len, decode_tokens, warmup=3):
    ids = []
    for s in range(batch):
        doc, _ = make_retrieval_doc(20, seq_len // 10, seed=1000 + s)
        t = tokenizer(doc).input_ids
        while len(t) < seq_len:
            t = t + t
        ids.append(t[:seq_len])
    input_ids = torch.tensor(ids, device=model.device)
    cuda = input_ids.is_cuda
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
    prefix, logits = prefill_and_compress(model, input_ids, cfg)
    nxt = logits.argmax(-1, keepdim=True)
    del logits
    row = {**describe(cfg), "batch": batch, "seq_len": seq_len, "kv_bytes_pct": prefix.stats["kv_bytes_pct"],
           "kv_MiB_logical": prefix.stats["kv_bytes"] / 2**20,
           "prefill_compress_ms": 1000 * prefix.stats["prefill_compress_s"]}
    if cuda:
        torch.cuda.synchronize()
        row["kv_MiB_measured"] = (torch.cuda.memory_allocated() - base) / 2**20
        row["prefill_peak_MiB"] = (torch.cuda.max_memory_allocated() - base) / 2**20
        torch.cuda.reset_peak_memory_stats()
    cache, pos, times = prefix.make_cache(), seq_len, []
    for step in range(decode_tokens + warmup):
        if cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        nxt = extend(model, cache, nxt, pos)[:, -1].argmax(-1, keepdim=True)
        if cuda:
            torch.cuda.synchronize()
        if step >= warmup:
            times.append(time.perf_counter() - t0)
        pos += 1
    row["decode_ms_per_token"] = 1000 * sum(times) / len(times)
    if cuda:
        row["decode_peak_MiB"] = (torch.cuda.max_memory_allocated() - base) / 2**20
    del cache, prefix
    return row


In [ ]:
import matplotlib.pyplot as plt

# Fixed colour + marker per method family (marker shape is a second, colour-independent cue).
FAMILY_STYLE = {
    "attn-select": ("#2a78d6", "o"),
    "ilf-select": ("#eb6834", "s"),
    "attn-mixed": ("#1baf7a", "^"),
    "ilf-mixed": ("#eda100", "D"),
    "uniform-quant": ("#e87ba4", "v"),
}


def plot_family(row):
    return "uniform-quant" if row["kind"] == "quant" else row["family"]


def tradeoff_plot(df, y, ylabel, title, path=None):
    """Quality vs KV bytes kept: one line per method family, full cache as a reference line."""
    df = df.assign(plot_family=df.apply(plot_family, axis=1))
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    full = df[df["kind"] == "full"]
    if len(full):
        ax.axhline(full[y].mean(), color="#8a8a85", lw=1.5, ls="--", label="full cache")
    for fam, (color, marker) in FAMILY_STYLE.items():
        part = df[df["plot_family"] == fam].sort_values("kv_bytes_pct")
        if len(part):
            ax.plot(part["kv_bytes_pct"], part[y], color=color, marker=marker, lw=2, ms=8, label=fam)
    ax.set_xlabel("KV-cache bytes kept (% of fp16)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


def lambda_plot(df, y, ylabel, title, path=None):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for fam, (color, marker) in FAMILY_STYLE.items():
        part = df[df["family"] == fam].sort_values("lambda")
        if len(part):
            ax.plot(part["lambda"], part[y], color=color, marker=marker, lw=2, ms=8, label=fam)
    ax.set_xlabel("λ (ILF weight)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


## 10. MMLU: general reasoning
For loglikelihood tasks, the context minus its last token is compressed, and the last context token plus
the answer are scored on top of the compressed cache. Without this split, single-token answers would
be scored from uncompressed prefill logits, and compression would have no effect.
`MMLU_LIMIT` applies per subject.


In [ ]:
mmlu_df = run_benchmark_suite(model, tokenizer, METHODS, MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT, MAX_LEN)
save_table(mmlu_df, MODEL_ID, "mmlu")
mmlu_df[["method", "task", "value", "kv_bytes_pct", "unique_positions_frac"]]


## 11. LongBench: long-context tasks
Prompts longer than `MAX_INPUT_TOKENS` are middle-truncated (LongBench convention). Scores are the
official LongBench metrics (QA-F1 or retrieval accuracy).


In [ ]:
longbench_df = run_benchmark_suite(model, tokenizer, METHODS, LONGBENCH_TASKS, LONGBENCH_LIMIT, None, MAX_LEN)
save_table(longbench_df, MODEL_ID, "longbench")
longbench_avg = (longbench_df.groupby(["method", "family", "kind", "budget", "lambda"], sort=False)
                 .agg(score=("value", "mean"), **{c: (c, "mean") for c in STAT_COLUMNS}).reset_index())
display(longbench_df.pivot(index="method", columns="task", values="value").loc[[m.name for m in METHODS]])
tradeoff_plot(longbench_avg, "score", "LongBench score (mean over tasks)", f"LongBench · {MODEL_ID}",
              RESULTS_DIR / MODEL_ID.replace("/", "__") / "longbench_tradeoff.png")


## 12. Long-range retrieval with a reused prefix
This is the experiment closest to the proposal's motivation. A long log hides `RETRIEVAL_FACTS` secret
codes at evenly spaced depths. Each log is **compressed once, before any question is seen**, and then
the same compressed prefix answers a question about every fact. A method that keeps only the tokens one
query cares about will fail on the others. We report exact match and digit accuracy (the fraction of the 5 digits right, in position). Digit
accuracy gives partial credit, because Qwen tokenizes each digit separately and exact match drops to 0
quickly under aggressive, query-agnostic eviction. Accuracy is also broken down by the fact's depth in the document.


In [ ]:
docs = retrieval_docs(tokenizer, RETRIEVAL_DOCS, RETRIEVAL_FACTS, min(RETRIEVAL_TOKENS, MAX_LEN - 64))
retrieval_df = run_retrieval(model, tokenizer, METHODS, docs)
save_table(retrieval_df, MODEL_ID, "retrieval_raw")
retrieval_summary = summarize_retrieval(retrieval_df)
save_table(retrieval_summary, MODEL_ID, "retrieval")
display(retrieval_summary[["method", "accuracy", "digit_acc", "kv_bytes_pct", "unique_positions_frac", "mean_layer_jaccard"]])
display(retrieval_df.assign(depth_bin=pd.cut(retrieval_df["depth"], [0, .25, .5, .75, 1]))
        .pivot_table(index="method", columns="depth_bin", values="digit_acc", aggfunc="mean", observed=False)
        .loc[[m.name for m in METHODS]])
tradeoff_plot(retrieval_summary, "digit_acc", "retrieval digit accuracy", f"Reused-prefix retrieval · {MODEL_ID}",
              RESULTS_DIR / MODEL_ID.replace("/", "__") / "retrieval_tradeoff.png")


## 13. Memory and decode latency
The proposal's workload is batch 4 × 4096 tokens. For each method:
* **kv_bytes_pct / kv_MiB_logical**: stored KV bytes, including quantization metadata.
* **kv_MiB_measured**: GPU memory still allocated after prefill + compression, i.e. the real cache
  footprint (model weights excluded).
* **prefill_peak_MiB / decode_peak_MiB**: peak extra memory. Prefill builds the full cache before
  compressing, so its peak is about the same for every method. Decode peak includes the transient
  dequantized copy of one layer.
* **decode_ms_per_token**: mean wall time per decoding step (whole batch), CUDA-synchronized, after warm-up.
* **prefill_compress_ms**: prefill plus scoring plus compression time.


In [ ]:
bench_df = pd.DataFrame([benchmark_memory_latency(model, tokenizer, cfg, BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS)
                         for cfg in METHODS])
save_table(bench_df, MODEL_ID, "memory_latency")
bench_df.drop(columns=["family", "kind"]).round(2)


## 14. Diversity and the λ sweep
**Diversity at matched budgets.** For selection methods, how many distinct token positions survive in
*at least one* layer (`unique_positions_frac`), and how similar the layers' kept sets are
(`mean_layer_jaccard`; lower means more diverse). This directly tests whether ILF reduces token
over-concentration.

**λ sweep.** Retrieval accuracy and diversity as the ILF weight grows, at the first numeric budget.
On Qwen2.5-0.5B that budget (10%) leaves every method at chance (digit accuracy ≈ 0.10), so this sweep
shows the diversity effect but not an accuracy effect. E2 in §14b repeats it at budgets where methods can
differ, and extends it towards *more* cross-layer agreement.


In [ ]:
diversity = retrieval_summary[retrieval_summary["kind"].isin(["select", "mixed"])]
display(diversity[["method", "budget", "lambda", "unique_positions", "unique_positions_frac",
                   "mean_layer_jaccard", "tokens_retained_frac", "accuracy", "digit_acc"]])

sweep_budget = next(b for b in BUDGETS if not isinstance(b, str))
sweep_methods = [make_config(f"ilf-{kind}@{budget_tag(sweep_budget)} λ={lam}", kind, sweep_budget, lam)
                 for kind in ("select", "mixed") for lam in LAMBDA_SWEEP]
sweep_df = summarize_retrieval(run_retrieval(model, tokenizer, sweep_methods, docs))
save_table(sweep_df, MODEL_ID, "lambda_sweep")
display(sweep_df[["method", "lambda", "accuracy", "digit_acc", "unique_positions_frac", "mean_layer_jaccard", "kv_bytes_pct"]])
out = RESULTS_DIR / MODEL_ID.replace("/", "__")
lambda_plot(sweep_df, "digit_acc", "retrieval digit accuracy", f"λ sweep at {budget_tag(sweep_budget)} budget", out / "lambda_accuracy.png")
lambda_plot(sweep_df, "unique_positions_frac", "unique positions kept (fraction)",
            f"Diversity vs λ at {budget_tag(sweep_budget)} budget", out / "lambda_diversity.png")


## 14b. Follow-up experiments: why ILF does not help, and what does
The first full run (Qwen2.5-0.5B) showed that ILF raises cross-layer diversity as designed but lowers
retrieval, that any eviction costs far more than quantization, and that the λ sweep above sits at a budget
where every method is at chance. The experiments below test four explanations. Each one is independent,
has its own `RUN_E*` flag in §2, saves `e*_*.csv` and PNGs next to the other results, and needs only
§1–§9 (you can skip §10–§14 when you only want these).

| | Question | Hypothesis tested |
|---|---|---|
| **E0** | At which budgets can methods differ at all? | sets the budgets for the rest |
| **E1** | Is a token that is kept in only a few layers worth anything? | H1: retrieval needs a fact in most layers, so spreading tokens over layers (ILF) cannot help |
| **E2** | Does retrieval rise with cross-layer agreement? | H1, as a sweep from ILF (diverse) to one shared kept set (consistent) |
| **E3** | What must the scorer see, and does ILF help when a cache built for one question serves another? | H2: the last-32-token window carries no information about future questions |
| **E4** | Is the collapse caused by the repeated filler text? | H3: the benchmark is unusually hostile to attention-based selection |
| **E5** | Fewer tokens at high precision, or more tokens at low precision? | H4: precision is cheap, eviction is expensive |
| **E6** | Does diversity over positions or content help where diversity over layers does not? | segment-balanced selection and a token-rarity (tf-idf) prior |

**Metrics.** Besides the free-running exact match used in §12, every question is also scored *teacher-forced*
in one forward pass: `gold_nll` is the mean negative log-likelihood of the gold digit tokens (chance is
ln 10 ≈ 2.30 nats when each digit is one token, as in Qwen), and `tf_exact` / `tf_digit_acc` say whether the
greedy prediction is right at every / each digit given the gold digits before it. These stay informative
where exact match is 0, and they are several times cheaper than decoding, so E1–E6 use them by default
(`EXP_GENERATE = True` adds decoding). `needle_kept_frac` is the share of (layer, fact-token) pairs that
survive compression and `code_layers_full` the number of layers that still hold a fact's whole code.


In [ ]:
# ---- Shared setup for E0-E6: documents, arm builders, plot helpers, sanity checks ----
import matplotlib.pyplot as plt

EXP_OUT = RESULTS_DIR / MODEL_ID.replace("/", "__")
EXP_OUT.mkdir(parents=True, exist_ok=True)
N_LAYERS = model_dims(model)[0]
CHANCE_NLL = math.log(10)  # uniform guess over ten digits, per digit token
EXP_TOKENS = min(RETRIEVAL_TOKENS, MAX_LEN - 64)
exp_docs = retrieval_docs(tokenizer, EXP_DOCS, RETRIEVAL_FACTS, EXP_TOKENS)
_natural_docs = None


def natural_docs():
    """The same facts hidden in non-repeating prose (built on first use; downloads WikiText once)."""
    global _natural_docs
    if _natural_docs is None:
        _natural_docs = retrieval_docs(tokenizer, EXP_DOCS, RETRIEVAL_FACTS, EXP_TOKENS, haystack="natural")
    return _natural_docs


def select_cfg(label, budget, lam=0.0, **kw):
    return make_config(f"{label}@{budget_tag(budget)}", "select", budget, lam, **kw)


def reference_arms(tags=None, bits=(8, 4)):
    return ([Arm(make_config("full", "full"), tags=tags)]
            + [Arm(make_config(f"quant-int{b}", "quant", quant_bits=b), tags=tags) for b in bits])


def full_value(summary, metric):
    return summary.loc[summary["kind"] == "full", metric].mean()


# ---- Oracle plans for E1 (they use the known token positions of the facts) ----
def knockout_plan(scheme, m, seed=0):
    """Evict every fact sentence from some layers of an otherwise untouched cache.
    "lowest" / "highest": the facts stay only in the m lowest / highest layers.
    "random_fact": in m random layers, drawn once per fact.  "random_token": drawn per token.
    "single": the facts are evicted from layer m only."""
    def plan(ctx):
        n_layers = ctx["n_layers"]
        override = torch.full((n_layers, 1, ctx["n_tokens"]), -1, dtype=torch.int16)
        rng = random.Random(100_003 * seed + ctx["doc"])
        for tokens in ctx["fact_tokens"]:
            shared = rng.sample(range(n_layers), m) if scheme == "random_fact" else None
            for t in tokens:
                if scheme == "lowest":
                    keep = range(m)
                elif scheme == "highest":
                    keep = range(n_layers - m, n_layers)
                elif scheme == "random_fact":
                    keep = shared
                elif scheme == "random_token":
                    keep = rng.sample(range(n_layers), m)
                elif scheme == "single":
                    keep = [l for l in range(n_layers) if l != m]
                else:
                    raise ValueError(scheme)
                drop = sorted(set(range(n_layers)) - set(keep))
                if drop:
                    override[drop, 0, t] = 0
        return {"bit_override": override.to(ctx["device"])}
    return plan


def protect_plan(ctx):
    """Rank every fact token first in every layer; the rest of the budget is filled by the scores."""
    mask = torch.zeros(1, ctx["n_tokens"], dtype=torch.bool)
    for tokens in ctx["fact_tokens"]:
        mask[0, tokens] = True
    return {"protect": mask.to(ctx["device"])}


# ---- Plot helpers. Colours are assigned in this fixed order and never cycled (at most 8 series). ----
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
SERIES_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
ORDINAL_BLUES = ["#86b6ef", "#2a78d6", "#104281"]  # one hue, light to dark, for ordered series
INK_SECONDARY, INK_MUTED, SURFACE = "#52514e", "#898781", "#fcfcfb"


def draw_series(ax, df, x, y, group, order=None, refs=(), colors=SERIES_COLORS):
    """One line per value of `group`; `refs` are (value, label) horizontal reference lines."""
    names = list(order) if order is not None else list(dict.fromkeys(df[group]))
    assert len(names) <= len(colors), "too many series for one chart: facet or fold some together"
    for k, name in enumerate(names):
        part = df[df[group] == name].sort_values(x)
        if len(part):
            ax.plot(part[x], part[y], color=colors[k], marker=SERIES_MARKERS[k], lw=2, ms=7, label=str(name))
    draw_refs(ax, refs)


def draw_refs(ax, refs):
    for value, label in refs:
        if value == value:  # skip NaN
            ax.axhline(value, color=INK_MUTED, lw=1.2, ls="--", zorder=1)
            ax.text(0.01, value, label, transform=ax.get_yaxis_transform(), va="bottom", ha="left",
                    fontsize=8, color=INK_SECONDARY)


def style_axis(ax, xlabel, ylabel, title, legend=True, unit=False):
    """`unit=True` fixes the y-axis to [0, 1] (accuracies), so panels and figures are comparable."""
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(alpha=0.25, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    if unit:
        ax.set_ylim(-0.03, 1.05)
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=8)


def save_show(fig, name):
    fig.tight_layout()
    fig.savefig(EXP_OUT / f"{name}.png", dpi=150)
    plt.show()


def series_plot(df, x, y, group, xlabel, ylabel, title, name, refs=(), order=None, unit=False):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    draw_series(ax, df, x, y, group, order, refs)
    style_axis(ax, xlabel, ylabel, title, unit=unit)
    save_show(fig, name)


def budget_title(b):
    return f"{budget_tag(b)} of bytes kept" if not isinstance(b, str) else f"bytes of uniform {b[3:]}-bit"


def panels(n, width=5.6, height=4.2):
    fig, axes = plt.subplots(1, n, figsize=(width * n, height), sharey=True, squeeze=False)
    return fig, list(axes[0])


def grouped_bars(ax, table, colors, refs=()):
    """`table`: index = x categories, columns = series. Bars of one category sit side by side with a gap."""
    n = len(table.columns)
    assert n <= len(colors), "too many series for one chart"
    width = 0.8 / n
    for k, col in enumerate(table.columns):
        ax.bar([i + (k - (n - 1) / 2) * width for i in range(len(table))], table[col], width * 0.88,
               color=colors[k], label=str(col), zorder=2)
    ax.set_xticks(range(len(table)))
    ax.set_xticklabels(table.index, fontsize=8)
    draw_refs(ax, refs)


# ---- Sanity checks of the new code paths (cheap) ----
_doc, _facts = exp_docs[0]
_ids, _spans = encode_doc(tokenizer, _doc, _facts, model.device)
_q, _gold, _ = question_ids(tokenizer, _facts[0], model.device)
print("fact 0 tokens:", tokenizer.convert_ids_to_tokens(_ids[0, _spans[0][0]].tolist()),
      "| code:", tokenizer.convert_ids_to_tokens(_ids[0, _spans[0][1]].tolist()),
      "| gold answer:", tokenizer.convert_ids_to_tokens(_gold[0].tolist()))

# 1. A scoring suffix must leave the prefix cache unchanged (attention is causal).
_full = make_config("full", "full")
_p0, _ = prefill_and_compress(model, _ids, _full)
_p1, _ = prefill_and_compress(model, _ids, _full, score_suffix_ids=_q)
_diff = (extend(model, _p0.make_cache(), _q, _p0.length, all_logits=True)
         - extend(model, _p1.make_cache(), _q, _p1.length, all_logits=True)).abs().max().item()
print(f"1. max |logit difference|, full cache with vs without a scoring suffix: {_diff:.2e}")

# 2. layer_mix = 1 keeps the same tokens in every layer; layer_mix = 0 is plain attention selection.
_cons, _ = prefill_and_compress(model, _ids, select_cfg("consistent", 0.5, score_mode="mix", layer_mix=1.0))
_attn, _ = prefill_and_compress(model, _ids, select_cfg("attention", 0.5))
_mix0, _ = prefill_and_compress(model, _ids, select_cfg("mix0", 0.5, score_mode="mix", layer_mix=0.0))
assert abs(_cons.stats["mean_layer_jaccard"] - 1) < 1e-6
print(f"2. consistent selection: layer Jaccard {_cons.stats['mean_layer_jaccard']:.3f}; "
      f"mix(0) agrees with plain attention on {((_attn.bits > 0) == (_mix0.bits > 0)).float().mean().item():.4f} of entries")

# 3. Knock-out plans: facts kept in all layers = untouched cache; kept in none = gone from every layer.
_ctx = {"doc": 0, "n_tokens": _ids.shape[1], "n_layers": N_LAYERS, "device": _ids.device,
        "fact_tokens": [s for s, _ in _spans]}
_all, _ = prefill_and_compress(model, _ids, _full, **knockout_plan("lowest", N_LAYERS)(_ctx))
_none, _ = prefill_and_compress(model, _ids, _full, **knockout_plan("lowest", 0)(_ctx))
assert needle_stats(_all.bits, *_spans[0])["needle_kept_frac"] == 1.0
assert needle_stats(_none.bits, *_spans[0])["needle_kept_frac"] == 0.0
print("3. knock-out plans evict exactly the fact tokens:", int((_none.bits == 0).sum()), "entries =",
      N_LAYERS, "layers x", len({t for s, _ in _spans for t in s}), "fact tokens")

# 4. Oracle protection keeps the facts in every layer at an unchanged byte budget.
_orc, _ = prefill_and_compress(model, _ids, select_cfg("oracle", 0.2), **protect_plan(_ctx))
_ref, _ = prefill_and_compress(model, _ids, select_cfg("attention", 0.2))
print(f"4. oracle-keep @20%: needle kept {needle_stats(_orc.bits, *_spans[0])['needle_kept_frac']:.2f} "
      f"(plain attention {needle_stats(_ref.bits, *_spans[0])['needle_kept_frac']:.2f}); "
      f"KV bytes {_orc.stats['kv_bytes_pct']:.2f}% vs {_ref.stats['kv_bytes_pct']:.2f}%")
del _p0, _p1, _cons, _attn, _mix0, _all, _none, _orc, _ref


### E0. Where can methods differ? Budget sweep with better instruments
Attention and ILF selection over a grid of budgets, plus the `@int8` budget of the first run, on
`EXP_DOCS` documents with free-running decoding **and** the teacher-forced metrics. Outputs:
* accuracy and `gold_nll` against bytes kept, with the chance level drawn in;
* how much of each fact survives (`code_layers_full`), and where in the document each layer keeps tokens;
* a paired attention-vs-ILF comparison at every budget (bootstrap over documents);
* `PROBE_BUDGETS` for E2, E3 and E6: budgets where plain attention selection is between 20% and 80% of
  the full-cache accuracy. Outside that band methods cannot be told apart.


In [ ]:
if RUN_E0:
    e0_budgets = [*KEEP_GRID, "int8"]
    e0_arms = reference_arms() + [Arm(make_config(f"{fam}@{budget_tag(b)}", "select", b, lam))
                                  for b in e0_budgets for fam, lam in (("attn-select", 0.0), ("ilf-select", LAMBDA))]
    e0_retention = {}
    e0_raw = run_arms(model, tokenizer, e0_arms, exp_docs, generate=True, retention=e0_retention)
    save_table(e0_raw, MODEL_ID, "e0_budget_sweep_raw")
    e0 = summarize_arms(e0_raw)
    save_table(e0, MODEL_ID, "e0_budget_sweep")
    display(e0[["method", "kv_bytes_pct", "correct", "digit_acc", "tf_exact", "gold_nll", "needle_kept_frac",
                "code_layers_full", "mean_layer_jaccard"]].round(3))

    tradeoff_plot(e0, "correct", "retrieval exact match (free-running)", f"E0 · exact match vs budget · {MODEL_ID}",
                  EXP_OUT / "e0_exact.png")
    e0_sel = e0[e0["kind"] == "select"]
    series_plot(e0_sel, "kv_bytes_pct", "gold_nll", "family", "KV-cache bytes kept (% of fp16)",
                "gold-code NLL per digit (nats, lower is better)", "E0 · likelihood of the right code", "e0_gold_nll",
                refs=[(CHANCE_NLL, "chance"), (full_value(e0, "gold_nll"), "full cache")])
    series_plot(e0_sel, "kv_bytes_pct", "code_layers_full", "family", "KV-cache bytes kept (% of fp16)",
                f"layers (of {N_LAYERS}) that still hold a fact's whole code", "E0 · how much of a fact survives",
                "e0_code_layers")

    # Where in the document does each layer keep tokens? (middle budget of the grid)
    mid = budget_tag(KEEP_GRID[len(KEEP_GRID) // 2])
    fig, axes = panels(2, width=5.2)
    for ax, fam in zip(axes, ("attn-select", "ilf-select")):
        heat = torch.stack(e0_retention[f"{fam}@{mid}"]).mean(0).numpy()
        im = ax.imshow(heat, aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1,
                       extent=(0, 1, 0, N_LAYERS))
        style_axis(ax, "depth in the document", "layer" if ax is axes[0] else "", f"{fam}@{mid}", legend=False)
        ax.grid(False)
    fig.colorbar(im, ax=axes, label="share of tokens kept", fraction=0.04, pad=0.02)
    fig.savefig(EXP_OUT / "e0_retention_by_depth.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Paired comparison at every budget: ILF minus attention on the same questions.
    e0_paired = pd.DataFrame([paired_compare(e0_raw, f"attn-select@{budget_tag(b)}", f"ilf-select@{budget_tag(b)}", m)
                              for b in e0_budgets for m in ("correct", "tf_exact", "gold_nll")])
    save_table(e0_paired, MODEL_ID, "e0_paired_ilf_vs_attention")
    display(e0_paired.round(3))

    # Regression against the first run (documents 0-9, free-running exact match).
    if MODEL_ID == "Qwen/Qwen2.5-0.5B" and not SMOKE_TEST:
        first_ten = e0_raw[e0_raw["doc"] < 10].groupby("method")["correct"].mean()
        for name, old in (("full", 0.905), ("quant-int4", 0.835), ("attn-select@int8", 0.150), ("ilf-select@int8", 0.075)):
            print(f"regression {name:>18}: now {first_ten[name]:.3f}, first run {old:.3f}")

    # Budgets where plain attention selection is neither at the floor nor at the ceiling.
    full_acc = full_value(e0, "tf_exact")
    grid = e0[(e0["family"] == "attn-select") & (e0["budget"] != "int8")].assign(b=lambda t: t["budget"].astype(float))
    usable = [float(b) for b in grid[(grid["tf_exact"] > 0.2 * full_acc) & (grid["tf_exact"] < 0.8 * full_acc)]["b"]]
    print(f"full-cache tf-exact {full_acc:.3f}; attention selection is in the 20-80% band at budgets {usable}")
    if not usable:  # nothing inside the band: take the budget closest to half of the full-cache accuracy
        usable = [float(grid.loc[(grid["tf_exact"] - 0.5 * full_acc).abs().idxmin(), "b"])]
        print(f"no grid budget falls in the band (widen KEEP_GRID); using the closest one, {usable}")
    suggested = sorted({min(usable), max(usable)})
    if AUTO_PROBE_BUDGETS:
        PROBE_BUDGETS = suggested
    print("PROBE_BUDGETS =", PROBE_BUDGETS)


### E1. Layer coverage: is a token kept in only a few layers worth anything?
ILF trades *layers per token* for *more distinct tokens*. This experiment measures what that trade
costs, using the known positions of the facts (an oracle, so not a compression method):
* **Knock-out.** The cache is left complete except that every fact sentence is evicted from all but `m`
  layers. The surviving layers are the lowest `m`, the highest `m`, `m` random layers per fact, or `m`
  random layers per token (the closest picture of what ILF produces).
* **Single-layer scan.** The facts are evicted from one layer at a time, which shows which layers carry retrieval.
* **Oracle-keep.** Selection at 10% and 20% bytes with the fact tokens ranked first in every layer. This is
  the ceiling for any query-agnostic token selector at that budget; the gap to plain attention selection is
  what a better scoring signal could recover.

If accuracy needs the fact in most layers, counting distinct positions kept in *any* layer
(`unique_positions`) measures the wrong thing, and the useful statistic is layers per token.


In [ ]:
if RUN_E1:
    m_grid = sorted({round(N_LAYERS * f) for f in (0, 1 / 12, 1 / 6, 1 / 3, 1 / 2, 2 / 3, 5 / 6, 1)})
    e1_arms = [Arm(make_config("full", "full"), tags={"experiment": "reference", "scheme": "full"})]
    for scheme, seeds in (("lowest", [0]), ("highest", [0]), ("random_fact", [0, 1]), ("random_token", [0])):
        for m in m_grid:
            for seed in seeds:
                e1_arms.append(Arm(make_config(f"knockout-{scheme}-m{m}-s{seed}", "full"), plan=knockout_plan(scheme, m, seed),
                                   tags={"experiment": "knockout", "scheme": scheme, "layers_kept": m, "seed": seed}))
    for layer in range(N_LAYERS):
        e1_arms.append(Arm(make_config(f"knockout-layer{layer}", "full"), plan=knockout_plan("single", layer),
                           tags={"experiment": "single_layer", "scheme": "single", "layer": layer}))
    for b in (0.1, 0.2):
        e1_arms.append(Arm(select_cfg("attn-select", b), tags={"experiment": "oracle", "scheme": "plain attention"}))
        e1_arms.append(Arm(select_cfg("oracle-keep", b), plan=protect_plan,
                           tags={"experiment": "oracle", "scheme": "facts ranked first"}))
    e1_raw = run_arms(model, tokenizer, e1_arms, exp_docs, generate=EXP_GENERATE)
    save_table(e1_raw, MODEL_ID, "e1_layer_coverage_raw")
    e1_full = e1_raw[e1_raw["experiment"] == "reference"]
    full_refs = lambda metric: [(e1_full[metric].astype(float).mean(), "full cache")]

    knock = summarize_arms(e1_raw[e1_raw["experiment"] == "knockout"], keys=["scheme", "layers_kept"])
    save_table(knock, MODEL_ID, "e1_knockout")
    display(knock.pivot(index="layers_kept", columns="scheme", values="tf_exact").round(3))
    series_plot(knock, "layers_kept", "tf_exact", "scheme", f"layers (of {N_LAYERS}) that still hold the fact",
                "teacher-forced exact match", f"E1 · a fact must survive in how many layers? · {MODEL_ID}",
                "e1_knockout_exact", refs=full_refs("tf_exact"), unit=True)
    series_plot(knock, "layers_kept", "gold_nll", "scheme", f"layers (of {N_LAYERS}) that still hold the fact",
                "gold-code NLL per digit (nats, lower is better)", "E1 · likelihood of the right code",
                "e1_knockout_nll", refs=[(CHANCE_NLL, "chance"), *full_refs("gold_nll")])

    scan = summarize_arms(e1_raw[e1_raw["experiment"] == "single_layer"], keys=["layer"])
    save_table(scan, MODEL_ID, "e1_single_layer")
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    ax.plot(scan["layer"], scan["gold_nll"], color=SERIES_COLORS[0], marker="o", lw=2, ms=6)
    draw_refs(ax, full_refs("gold_nll"))
    style_axis(ax, "layer the facts are evicted from", "gold-code NLL per digit (nats)",
               "E1 · which single layer hurts most to lose", legend=False)
    save_show(fig, "e1_single_layer")

    oracle = summarize_arms(e1_raw[e1_raw["experiment"] == "oracle"], keys=["budget", "scheme"])
    save_table(oracle, MODEL_ID, "e1_oracle_keep")
    display(oracle[["budget", "scheme", "tf_exact", "tf_digit_acc", "gold_nll", "needle_kept_frac", "kv_bytes_pct"]].round(3))
    print(f"full cache: tf-exact {full_refs('tf_exact')[0][0]:.3f}, gold NLL {full_refs('gold_nll')[0][0]:.2f}")


### E2. Consistency sweep: from ILF to one shared kept set
The corrected λ sweep. At each probe budget, selection methods are lined up from most diverse to most
consistent across layers, at identical bytes:
* **ILF** with λ = 4, 1, 0.25, plus two repairs of its formula: `ilf-gated` (no boost for tokens that no
  layer would keep, which otherwise get the largest boost) and a tighter τ.
* **plain attention**, each layer on its own (the baseline).
* **mix α**: `(1−α)·rank_l + α·mean-rank over layers`; α = 1 keeps the same tokens in every layer.
* **lf**: `a·(1 + λ·log(1+d_i))`, the opposite of ILF, boosting tokens that many layers want.
* **random controls**: one random set shared by all layers vs an independent random set per layer. These two
  keep the same number of tokens and differ only in cross-layer agreement.

The figure plots each arm's retrieval against `mean_layer_jaccard`. H1 predicts a rising trend.


In [ ]:
E2_GROUPS = {"per-layer attention": (SERIES_COLORS[0], "o"), "ILF (more diverse)": (SERIES_COLORS[1], "s"),
             "more consistent": (SERIES_COLORS[2], "^"), "random control": (INK_MUTED, "X")}


def e2_arms_at(b):
    def arm(label, group, lam=0.0, **kw):
        return Arm(select_cfg(label, b, lam, **kw), tags={"arm": label, "group": group})
    diverse, consistent = "ILF (more diverse)", "more consistent"
    return ([arm(f"ilf λ={lam:g}", diverse, lam) for lam in (4.0, 1.0, 0.25)]
            + [arm(f"ilf-gated λ={lam:g}", diverse, lam, score_mode="ilf_gated") for lam in (4.0, 1.0)]
            + [arm("ilf-tight-τ λ=1", diverse, 1.0, tau_quantile=1 - b / 2)]
            + [arm("attention", "per-layer attention")]
            + [arm(f"mix α={a:g}", consistent, score_mode="mix", layer_mix=a) for a in (0.25, 0.5, 0.75, 1.0)]
            + [arm(f"lf λ={lam:g}", consistent, lam, score_mode="lf") for lam in (0.25, 1.0, 4.0)]
            + [arm("random-shared", "random control", score_mode="random_shared"),
               arm("random-per-layer", "random control", score_mode="random_layer")])


def scatter_by_group(ax, df, x, y):
    for group, (color, marker) in E2_GROUPS.items():
        part = df[df["group"] == group]
        ax.scatter(part[x], part[y], s=70, color=color, marker=marker, edgecolors=SURFACE, linewidths=1.5,
                   label=group, zorder=4 if group == "per-layer attention" else 3)
        for _, r in part.iterrows():  # every point is named; the table above carries the exact values
            ax.annotate(r["arm"], (r[x], r[y]), xytext=(4, 5), textcoords="offset points", fontsize=7,
                        color=INK_SECONDARY, rotation=35, ha="left", va="bottom")


if RUN_E2:
    e2_arms = [Arm(make_config("full", "full"), tags={"arm": "full", "group": "reference"})]
    for b in PROBE_BUDGETS:
        e2_arms += e2_arms_at(b)
    e2_raw = run_arms(model, tokenizer, e2_arms, exp_docs, generate=EXP_GENERATE)
    save_table(e2_raw, MODEL_ID, "e2_consistency_raw")
    e2 = summarize_arms(e2_raw, keys=["method", "arm", "group", "budget", "kind"])
    save_table(e2, MODEL_ID, "e2_consistency")
    display(e2[["method", "group", "mean_layer_jaccard", "unique_positions_frac", "tf_exact", "tf_digit_acc", "gold_nll",
                "code_layers_full"]].round(3))

    for metric, ylabel, refs in (
            ("tf_exact", "teacher-forced exact match", [(full_value(e2, "tf_exact"), "full cache")]),
            ("gold_nll", "gold-code NLL per digit (nats, lower is better)",
             [(CHANCE_NLL, "chance"), (full_value(e2, "gold_nll"), "full cache")])):
        fig, axes = panels(len(PROBE_BUDGETS))
        for ax, b in zip(axes, PROBE_BUDGETS):
            scatter_by_group(ax, e2[e2["budget"] == str(b)], "mean_layer_jaccard", metric)
            ax.margins(x=0.08, y=0.2)  # room for the point labels
            draw_refs(ax, refs)
            style_axis(ax, "cross-layer agreement of kept sets (mean Jaccard)", ylabel if ax is axes[0] else "",
                       f"E2 · {budget_title(b)}", legend=ax is axes[0], unit=metric == "tf_exact")
        save_show(fig, f"e2_consistency_{metric}")

    # Every arm against plain attention on the same questions.
    e2_paired = pd.DataFrame([
        {"budget": b, **paired_compare(e2_raw, f"attention@{budget_tag(b)}", arm.cfg.name, metric)}
        for b in PROBE_BUDGETS for arm in e2_arms_at(b) if arm.tags["arm"] != "attention"
        for metric in ("tf_exact", "gold_nll")])
    save_table(e2_paired, MODEL_ID, "e2_paired_vs_attention")
    display(e2_paired[e2_paired["metric"] == "gold_nll"].round(3))


### E3. Scoring signal: what must the selector see?
ILF only re-weights the attention scores it is given, so this experiment varies where those scores come
from and keeps everything else fixed. The compressed cache always holds the document only; any extra
tokens are used for scoring and then dropped.

| Signal | Scores are the attention from … |
|---|---|
| `window` | the last 32 document tokens (what §12 does) |
| `all_queries` | every later document token, averaged over the tokens that can see each key (H2O-style) |
| `probe` | a fixed, task-agnostic instruction appended after the document |
| `reconstruct` | the document read a second time after "Repeat the text above", max over queries. A simplified version of KVzip's query-agnostic scoring ([arXiv 2505.23416](https://arxiv.org/abs/2505.23416)), not a reimplementation |
| question (same) | the very question that is then asked: query-aware compression, not reusable |
| question (different) | a question about *another* fact: a query-aware cache reused for a new question, the case the proposal is about |

Each signal is combined with plain attention selection, ILF (λ = `LAMBDA`) and the consistent selector
(one kept set for all layers). ILF's fairest test is the "question (different)" row.


In [ ]:
E3_SIGNAL_LABELS = {"window": "last 32\ntokens", "all_queries": "all prefix\nqueries", "probe": "generic\nprobe",
                    "reconstruct": "re-reading\n(KVzip-style)", "question (same)": "question\n(same)",
                    "question (different)": "question\n(different)"}

if RUN_E3:
    e3_signals = ["window", "all_queries", "probe", "reconstruct", "query"]
    longest = max(len(tokenizer(doc).input_ids) for doc, _ in exp_docs)
    if 2 * longest + 32 > getattr(model.config, "max_position_embeddings", 0):
        e3_signals.remove("reconstruct")
        print("Skipping 'reconstruct': two copies of the document do not fit the model's context.")
    e3_selectors = {"attention": {}, "ilf": {"lam": LAMBDA}, "consistent": {"score_mode": "mix", "layer_mix": 1.0}}
    e3_arms = [Arm(make_config("full", "full"), tags={"selector": "full"})]
    e3_arms += [Arm(select_cfg(f"{sel}/{sig}", b, **kw), scoring=sig, tags={"selector": sel})
                for b in PROBE_BUDGETS for sig in e3_signals for sel, kw in e3_selectors.items()]
    e3_raw = run_arms(model, tokenizer, e3_arms, exp_docs, generate=EXP_GENERATE)
    e3_raw["signal"] = e3_raw["scoring"]
    e3_raw.loc[e3_raw["query_match"] == True, "signal"] = "question (same)"       # noqa: E712
    e3_raw.loc[e3_raw["query_match"] == False, "signal"] = "question (different)"  # noqa: E712
    save_table(e3_raw, MODEL_ID, "e3_scoring_signal_raw")
    e3 = summarize_arms(e3_raw, keys=["budget", "signal", "selector", "kind"])
    save_table(e3, MODEL_ID, "e3_scoring_signal")
    e3_sel = e3[e3["kind"] == "select"]
    order = [s for s in E3_SIGNAL_LABELS if s in set(e3_sel["signal"])]
    for metric in ("tf_exact", "gold_nll", "code_layers_full"):
        print(metric)
        display(e3_sel.pivot_table(index=["budget", "signal"], columns="selector", values=metric)
                .reindex(columns=list(e3_selectors)).round(3))

    fig, axes = panels(len(PROBE_BUDGETS), width=6.4)
    for ax, b in zip(axes, PROBE_BUDGETS):
        table = (e3_sel[e3_sel["budget"] == str(b)].pivot(index="signal", columns="selector", values="tf_exact")
                 .reindex(index=order, columns=list(e3_selectors)))
        table.index = [E3_SIGNAL_LABELS[s] for s in table.index]
        grouped_bars(ax, table, SERIES_COLORS, refs=[(full_value(e3, "tf_exact"), "full cache")])
        style_axis(ax, "where the attention scores come from", "teacher-forced exact match" if ax is axes[0] else "",
                   f"E3 · {budget_title(b)}", legend=ax is axes[0], unit=True)
        ax.grid(axis="x", visible=False)
    save_show(fig, "e3_scoring_signal")


### E4. Haystack robustness: is the collapse caused by the repeated filler?
The §12 documents repeat twelve filler sentences a few hundred times. Repeated text may soak up the
scoring window's attention (each filler token has many earlier copies to attend to) and push the unique
fact tokens out of every layer's top-k. Here the same facts, at the same depths, are hidden in
non-repeating, digit-free Wikipedia prose (WikiText test split), and the budget sweep is repeated on both
haystacks for plain attention, ILF and consistent selection. If the cliff moves to much lower budgets on
natural prose, report both and treat the natural haystack as the primary benchmark.


In [ ]:
E4_FAMILIES = {"attn-select": {}, "ilf-select": {"lam": LAMBDA}, "consistent-select": {"score_mode": "mix", "layer_mix": 1.0}}

if RUN_E4:
    e4_budgets = sorted({0.1, 0.2, *KEEP_GRID})
    parts = []
    for haystack, docs in (("repeated filler", exp_docs), ("natural prose", natural_docs())):
        tags = {"haystack": haystack}
        arms = reference_arms(tags) + [Arm(select_cfg(fam, b, **kw), tags=tags)
                                       for b in e4_budgets for fam, kw in E4_FAMILIES.items()]
        print(f"--- {haystack}: {len(tokenizer(docs[0][0]).input_ids)} tokens in document 0 ---")
        parts.append(run_arms(model, tokenizer, arms, docs, generate=EXP_GENERATE))
    e4_raw = pd.concat(parts, ignore_index=True)
    save_table(e4_raw, MODEL_ID, "e4_haystack_raw")
    e4 = summarize_arms(e4_raw, by=["haystack"])
    e4["series"] = e4.apply(plot_family, axis=1)
    save_table(e4, MODEL_ID, "e4_haystack")
    display(e4.pivot_table(index=["series", "budget"], columns="haystack", values="tf_exact").round(3))

    for metric, ylabel in (("tf_exact", "teacher-forced exact match"),
                           ("gold_nll", "gold-code NLL per digit (nats, lower is better)")):
        fig, axes = panels(2)
        for ax, haystack in zip(axes, ("repeated filler", "natural prose")):
            part = e4[e4["haystack"] == haystack]
            refs = [(full_value(part, metric), "full cache")] + ([(CHANCE_NLL, "chance")] if metric == "gold_nll" else [])
            draw_series(ax, part[part["kind"] != "full"], "kv_bytes_pct", metric, "series",
                        order=[*E4_FAMILIES, "uniform-quant"], refs=refs)
            style_axis(ax, "KV-cache bytes kept (% of fp16)", ylabel if ax is axes[0] else "", f"E4 · {haystack}",
                       legend=ax is axes[0], unit=metric == "tf_exact")
        save_show(fig, f"e4_haystack_{metric}")


### E5. Where should the bytes go? Fewer tokens at high precision or more at low
In the first run uniform 4-bit quantization (32% of bytes, nothing evicted) beat every method that evicts
at the same bytes by a wide margin. This experiment varies only how a fixed byte budget is split over
precision tiers (`tier_split` = share of the budget for 16 / 8 / 4-bit tokens), from "few tokens at 16
bits" to "as many tokens as possible at 4 bits", for four rankings: plain attention, ILF, and two random
rankings (shared across layers, independent per layer). If the random rankings match the score-based ones,
the scores carry no information about which tokens deserve bits. Uniform 8 / 4 / 2-bit quantization is
included for reference; 2-bit (KIVI-style, packed four codes per byte) reaches about 19% of bytes
without evicting anything.


In [ ]:
E5_SPLITS = {"16-bit only": (1, 0, 0), "16/8/4 thirds": (1 / 3, 1 / 3, 1 / 3), "8-bit only": (0, 1, 0),
             "10% 16-bit, rest 4-bit": (0.1, 0, 0.9), "4-bit only": (0, 0, 1)}
E5_RANKINGS = {"attention": {}, "ilf": {"lam": LAMBDA}, "random-shared": {"score_mode": "random_shared"},
               "random-per-layer": {"score_mode": "random_layer"}}

if RUN_E5:
    e5_budgets = [0.1, 0.2, "int4"]
    e5_arms = reference_arms({"ranking": "uniform", "split": "uniform"}, bits=(8, 4, 2))
    e5_arms += [Arm(make_config(f"{rank}/{split}@{budget_tag(b)}", "mixed", b, tier_split=shares, **kw),
                    tags={"ranking": rank, "split": split})
                for b in e5_budgets for split, shares in E5_SPLITS.items() for rank, kw in E5_RANKINGS.items()]
    e5_raw = run_arms(model, tokenizer, e5_arms, exp_docs, generate=EXP_GENERATE)
    save_table(e5_raw, MODEL_ID, "e5_tier_split_raw")
    e5 = summarize_arms(e5_raw, by=["ranking", "split"])
    save_table(e5, MODEL_ID, "e5_tier_split")
    display(e5[e5["ranking"] == "uniform"][["method", "kv_bytes_pct", "tf_exact", "gold_nll"]].round(3))
    e5_mixed = e5[e5["kind"] == "mixed"]
    for metric in ("tokens_retained_frac", "tf_exact", "gold_nll"):
        print(metric)
        display(e5_mixed.pivot_table(index=["budget", "split"], columns="ranking", values=metric, sort=False).round(3))

    fig, axes = panels(len(e5_budgets), width=5.0)
    for ax, b in zip(axes, e5_budgets):
        part = e5_mixed[e5_mixed["budget"] == str(b)]
        draw_series(ax, part, "tokens_retained_frac", "tf_exact", "ranking", order=list(E5_RANKINGS),
                    refs=[(full_value(e5, "tf_exact"), "full cache")])
        for _, r in part[part["ranking"] == "attention"].iterrows():  # name the splits once, on one series
            ax.annotate(r["split"], (r["tokens_retained_frac"], r["tf_exact"]), xytext=(3, 7), rotation=35,
                        textcoords="offset points", ha="left", va="bottom", fontsize=7, color=INK_SECONDARY)
        style_axis(ax, "share of tokens kept (same bytes)", "teacher-forced exact match" if ax is axes[0] else "",
                   f"E5 · {budget_title(b)}", legend=ax is axes[0], unit=True)
    save_show(fig, "e5_tier_split")


### E6. Diversity over positions or content instead of layers
Two selectors that diversify along a different axis than ILF, compared with plain attention, ILF and
consistent selection on both haystacks:
* **segment**: the document is cut into `n_segments` equal blocks and every block keeps the same share of
  its tokens (ranked by attention inside the block). This targets the recency concentration seen in the
  first run, where almost only the last facts survived.
* **idf**: `a·(1 + λ·log(M / count))`, where `count` is how often the token's id occurs in the document. This
  is the literal tf-idf analogue: rare tokens are boosted.

Caveat for the write-up: on the repeated-filler haystack the fact tokens are almost the only rare tokens,
so `idf` is close to an oracle there by construction. Only its result on natural prose is evidence.


In [ ]:
E6_ARMS = {"attention": {}, "ilf λ=1": {"lam": 1.0}, "consistent": {"score_mode": "mix", "layer_mix": 1.0},
           "segment": {"score_mode": "segment"}, "idf λ=0.5": {"score_mode": "idf", "lam": 0.5},
           "idf λ=1": {"score_mode": "idf", "lam": 1.0}, "idf λ=2": {"score_mode": "idf", "lam": 2.0}}

if RUN_E6:
    e6_budgets = sorted({0.2, *PROBE_BUDGETS})[:3]
    parts = []
    for haystack, docs in (("repeated filler", exp_docs), ("natural prose", natural_docs())):
        arms = [Arm(make_config("full", "full"), tags={"haystack": haystack, "arm": "full"})]
        arms += [Arm(select_cfg(label, b, **kw), tags={"haystack": haystack, "arm": label})
                 for b in e6_budgets for label, kw in E6_ARMS.items()]
        print(f"--- {haystack} ---")
        parts.append(run_arms(model, tokenizer, arms, docs, generate=EXP_GENERATE))
    e6_raw = pd.concat(parts, ignore_index=True)
    save_table(e6_raw, MODEL_ID, "e6_content_diversity_raw")
    e6 = summarize_arms(e6_raw, keys=["haystack", "arm", "budget", "kind"])
    save_table(e6, MODEL_ID, "e6_content_diversity")
    e6_sel = e6[e6["kind"] == "select"]
    for metric in ("tf_exact", "gold_nll", "code_layers_full"):
        print(metric)
        display(e6_sel.pivot_table(index=["haystack", "arm"], columns="budget", values=metric, sort=False).round(3))

    fig, axes = panels(2, width=6.4)
    for ax, haystack in zip(axes, ("repeated filler", "natural prose")):
        part = e6_sel[e6_sel["haystack"] == haystack]
        table = part.pivot(index="arm", columns="budget", values="tf_exact").reindex(
            index=list(E6_ARMS), columns=[str(b) for b in e6_budgets])
        table.columns = [f"{budget_tag(b)} of bytes" for b in e6_budgets]
        grouped_bars(ax, table, ORDINAL_BLUES,
                     refs=[(full_value(e6[e6["haystack"] == haystack], "tf_exact"), "full cache")])
        style_axis(ax, "selection score", "teacher-forced exact match" if ax is axes[0] else "", f"E6 · {haystack}",
                   legend=ax is axes[0], unit=True)
        ax.grid(axis="x", visible=False)
        ax.tick_params(axis="x", labelrotation=20)
    save_show(fig, "e6_content_diversity")


## 15. Summary table
One row per method, joining every experiment run above for the current model.


In [ ]:
def summary_table(mmlu_df, longbench_df, retrieval_summary, bench_df):
    s = retrieval_summary[["method", "kv_bytes_pct", "unique_positions_frac", "mean_layer_jaccard", "accuracy", "digit_acc"]]
    s = s.rename(columns={"accuracy": "retrieval_exact", "digit_acc": "retrieval_digit_acc"})
    s = s.merge(mmlu_df.groupby("method", sort=False)["value"].mean().rename("mmlu_acc").reset_index(), on="method")
    s = s.merge(longbench_df.groupby("method", sort=False)["value"].mean().rename("longbench").reset_index(), on="method")
    cols = ["method"] + [c for c in ["kv_MiB_measured", "decode_ms_per_token"] if c in bench_df]
    return s.merge(bench_df[cols], on="method")


summary = summary_table(mmlu_df, longbench_df, retrieval_summary, bench_df)
save_table(summary, MODEL_ID, "summary")
summary.round(3)


## 16. Other models (Qwen 2.5 sizes, Llama 2, Llama 3)
Runs the whole suite for each model in `EXTRA_MODEL_IDS`, freeing GPU memory in between. Large models
need a bigger GPU (an A100 for 7–8B with 4k contexts) and a lot more time, so you may want to shrink
`METHODS` or the limits first. Llama-2's 4096-token context limits `MAX_LEN` automatically.


In [ ]:
RUN_EXTRA_MODELS = False

def run_suite(model_id):
    mdl, tok, max_len = load_model(model_id)
    try:
        mm = run_benchmark_suite(mdl, tok, METHODS, MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT, max_len)
        lb = run_benchmark_suite(mdl, tok, METHODS, LONGBENCH_TASKS, LONGBENCH_LIMIT, None, max_len)
        dc = retrieval_docs(tok, RETRIEVAL_DOCS, RETRIEVAL_FACTS, min(RETRIEVAL_TOKENS, max_len - 64))
        rs = summarize_retrieval(run_retrieval(mdl, tok, METHODS, dc))
        bn = pd.DataFrame([benchmark_memory_latency(mdl, tok, c, BENCH_BATCH, min(BENCH_SEQ_LEN, max_len - BENCH_DECODE_TOKENS - 8),
                                                    BENCH_DECODE_TOKENS) for c in METHODS])
        for df, name in [(mm, "mmlu"), (lb, "longbench"), (rs, "retrieval"), (bn, "memory_latency")]:
            save_table(df, model_id, name)
        s = summary_table(mm, lb, rs, bn).assign(model=model_id)
        save_table(s, model_id, "summary")
        return s
    finally:
        del mdl
        gc.collect()
        torch.cuda.empty_cache()


if RUN_EXTRA_MODELS:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    all_summaries = pd.concat([summary.assign(model=MODEL_ID)] + [run_suite(m) for m in EXTRA_MODEL_IDS])
    all_summaries.to_csv(RESULTS_DIR / "summary_all_models.csv", index=False)
    display(all_summaries.round(3))


**Limitations to state in the write-up.** Batches are unpadded (equal-length inputs). Quantized tokens
are dequantized every decode step in PyTorch, not with a fused kernel, so the latency numbers include
that overhead. The prefill peak is not reduced, because compression happens after prefill. MMLU
contexts are short, so most of the prompt sits inside the protected window.

References: [Transformers cache guide](https://huggingface.co/docs/transformers/v4.57.1/en/cache_explanation),
[Attention interface](https://huggingface.co/docs/transformers/v4.57.1/en/attention_interface),
[LM Evaluation Harness API](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md),
[LongBench tasks](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/longbench/README.md),
SnapKV (Li et al., 2024), KIVI (Liu et al., 2024).
